In [1]:
"""
================================================================================
  09_WILLIE_FUSEG_CSD.ipynb — Cell 1
  Config + Paths + Manifest Loading + Architecture (MINI first)
  
  Professor's Direction:
    ✓ Replace SAM2 seg decoder → FUSegNet-style P-scSE decoder
    ✓ Keep ALL 3 tasks: C + S + D
    ✓ Combined multi-task accuracy → target 95%
    ✓ Train MINI / BASE / XL
    ✓ Publication visuals
  
  Architecture:
    DINOv2 → FPN → WA-CSA → [CLS(MoE) + SEG(P-scSE+FiLM) + DET(anchor-free)]
  
  Data: Loaded from existing CSV manifests (built in Notebook 01)
================================================================================
"""

import os, sys, time, math, json, ast, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast
from einops import rearrange, repeat

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────────
# 0. DEVICE + PROJECT PATHS
# ──────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"{'='*80}")
print(f"  09_WILLIE_FUSEG_CSD — Cell 1")
print(f"  {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"  GPU: {torch.cuda.get_device_name(0)} ({gpu.total_memory / 1e9:.1f} GB)")
print(f"  PyTorch: {torch.__version__}")

# Project root
ROOT = Path(".")

# Manifest paths (from Notebook 01 — DO NOT change)
CLS_MANIFEST_DIR = ROOT / "artifacts" / "woundshot_v2" / "manifests"
LOCKED_DIR       = ROOT / "artifacts" / "WoundShot_LOCKED_INPUTS" / "tables"

CLS_TRAIN_CSV = CLS_MANIFEST_DIR / "cls_train.csv"
CLS_VAL_CSV   = CLS_MANIFEST_DIR / "cls_val.csv"
CLS_TEST_CSV  = CLS_MANIFEST_DIR / "cls_test.csv"

SEG_TRAIN_CSV = LOCKED_DIR / "ws_seg_manifest_fuseg_train.csv"
SEG_VAL_CSV   = LOCKED_DIR / "ws_seg_manifest_fuseg_val.csv"

DET_TRAIN_CSV = LOCKED_DIR / "ws_det_manifest_yolo_train.csv"
DET_VAL_CSV   = LOCKED_DIR / "ws_det_manifest_yolo_val.csv"

# Output directory for this notebook
CKPT_DIR = ROOT / "artifacts" / "09_fuseg_csd"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Classes
CLASS_NAMES = ["diabetic", "no_wound", "pressure", "surgical", "venous"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
NUM_CLASSES = 5

# ──────────────────────────────────────────────────────────────────────────────
# 1. LOAD & VERIFY ALL MANIFESTS
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  📂 Loading CSV Manifests")
print(f"{'─'*80}")

all_csvs = {
    "cls_train": CLS_TRAIN_CSV,
    "cls_val":   CLS_VAL_CSV,
    "cls_test":  CLS_TEST_CSV,
    "seg_train": SEG_TRAIN_CSV,
    "seg_val":   SEG_VAL_CSV,
    "det_train": DET_TRAIN_CSV,
    "det_val":   DET_VAL_CSV,
}

MANIFESTS = {}
for name, path in all_csvs.items():
    assert path.exists(), f"❌ Not found: {path}"
    df = pd.read_csv(path)
    MANIFESTS[name] = df
    print(f"  ✅ {name:12s}: {len(df):>5} rows  ← {path.name}")

# ── Classification manifest details ──
cls_df = MANIFESTS["cls_train"]

# Identify correct columns
IMG_COL = [c for c in cls_df.columns if "image" in c.lower() or "path" in c.lower()][0]
CLS_LABEL_COL = "unified_class"  # 5 classes: diabetic, pressure, surgical, venous, no_wound
CLS_INT_COL = "unified_label"    # integer label

print(f"\n  Classification columns: img={IMG_COL}, class={CLS_LABEL_COL}, label={CLS_INT_COL}")
print(f"  Train distribution:")
for cls_name in sorted(cls_df[CLS_LABEL_COL].unique()):
    n = (cls_df[CLS_LABEL_COL] == cls_name).sum()
    print(f"    {cls_name:15s}: {n:4d}")

# ── Segmentation manifest details ──
seg_df = MANIFESTS["seg_train"]
SEG_IMG_COL  = [c for c in seg_df.columns if c in ["img", "image_path", "image"]][0]
SEG_MASK_COL = [c for c in seg_df.columns if c in ["mask", "mask_path"]][0]
print(f"\n  Segmentation columns: img={SEG_IMG_COL}, mask={SEG_MASK_COL}")

# ── Detection manifest details ──
det_df = MANIFESTS["det_train"]
DET_IMG_COL   = [c for c in det_df.columns if c in ["img", "image_path", "image"]][0]
DET_LABEL_COL = [c for c in det_df.columns if c in ["label", "label_path", "bbox_yolo"]][0]
print(f"  Detection columns: img={DET_IMG_COL}, bbox={DET_LABEL_COL}")

# ── Summary ──
print(f"\n  📊 Data Summary:")
print(f"    Classification: {len(MANIFESTS['cls_train'])} train + {len(MANIFESTS['cls_val'])} val + {len(MANIFESTS['cls_test'])} test = {len(MANIFESTS['cls_train'])+len(MANIFESTS['cls_val'])+len(MANIFESTS['cls_test'])} total")
print(f"    Segmentation:   {len(MANIFESTS['seg_train'])} train + {len(MANIFESTS['seg_val'])} val = {len(MANIFESTS['seg_train'])+len(MANIFESTS['seg_val'])} total")
print(f"    Detection:       {len(MANIFESTS['det_train'])} train + {len(MANIFESTS['det_val'])} val = {len(MANIFESTS['det_train'])+len(MANIFESTS['det_val'])} total")

# ──────────────────────────────────────────────────────────────────────────────
# 2. MODEL CONFIGURATIONS
# ──────────────────────────────────────────────────────────────────────────────
@dataclass
class ModelConfig:
    name: str = "MINI"
    backbone_name: str = "dinov2_vits14"
    backbone_dim: int = 384
    backbone_layers: List[int] = field(default_factory=lambda: [2, 5, 8, 11])
    img_size: int = 518           # DINOv2 native: 518 = 37×14
    patch_size: int = 14
    fpn_dim: int = 256
    fpn_levels: int = 4
    wa_csa_layers: int = 1
    wa_csa_heads: int = 8
    wa_csa_dropout: float = 0.1
    num_classes: int = 5
    num_experts: int = 2
    top_k_experts: int = 2
    cls_embed_dim: int = 128      # WTCS bridge dim
    seg_out_channels: int = 1
    pscse_reduction: int = 16
    det_max_objects: int = 20
    freeze_backbone_epochs: int = 3
    
    @property
    def grid_size(self) -> int:
        return self.img_size // self.patch_size


def get_model_config(variant: str = "MINI") -> ModelConfig:
    configs = {
        "MINI": ModelConfig(
            name="MINI", backbone_name="dinov2_vits14", backbone_dim=384,
            fpn_dim=256, wa_csa_layers=1, num_experts=2, top_k_experts=2,
        ),
        "BASE": ModelConfig(
            name="BASE", backbone_name="dinov2_vitl14", backbone_dim=1024,
            fpn_dim=256, wa_csa_layers=2, num_experts=4, top_k_experts=2,
        ),
        "XL": ModelConfig(
            name="XL", backbone_name="dinov2_vitl14", backbone_dim=1024,
            fpn_dim=384, wa_csa_layers=4, num_experts=8, top_k_experts=2,
        ),
    }
    return configs[variant]


# ──────────────────────────────────────────────────────────────────────────────
# 3. DINOv2 MULTI-SCALE ENCODER
# ──────────────────────────────────────────────────────────────────────────────
class DINOv2MultiScale(nn.Module):
    """Shared DINOv2 backbone with hooks at layers [2,5,8,11]."""
    
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.features = {}
        self.backbone = torch.hub.load("facebookresearch/dinov2", cfg.backbone_name, pretrained=True)
        self._hook_handles = []
        for idx in cfg.backbone_layers:
            h = self.backbone.blocks[idx].register_forward_hook(self._make_hook(idx))
            self._hook_handles.append(h)
        self.set_frozen(True)
    
    def _make_hook(self, idx):
        def hook(mod, inp, out):
            tokens = out[:, 1:, :]
            G = self.cfg.grid_size
            self.features[idx] = rearrange(tokens, "b (h w) d -> b d h w", h=G, w=G)
        return hook
    
    def set_frozen(self, frozen):
        for p in self.backbone.parameters():
            p.requires_grad = not frozen
    
    def forward(self, x):
        self.features.clear()
        if x.shape[-1] != self.cfg.img_size:
            x = F.interpolate(x, size=self.cfg.img_size, mode="bilinear", align_corners=False)
        self.backbone(x)
        return [self.features[idx] for idx in self.cfg.backbone_layers]


# ──────────────────────────────────────────────────────────────────────────────
# 4. FEATURE PYRAMID NECK
# ──────────────────────────────────────────────────────────────────────────────
class FeaturePyramidNeck(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        D, F = cfg.backbone_dim, cfg.fpn_dim
        self.laterals = nn.ModuleList([
            nn.Sequential(nn.Conv2d(D, F, 1, bias=False), nn.GroupNorm(32, F), nn.GELU())
            for _ in range(cfg.fpn_levels)
        ])
        self.smooth = nn.ModuleList([
            nn.Sequential(nn.Conv2d(F, F, 3, padding=1, bias=False), nn.GroupNorm(32, F), nn.GELU())
            for _ in range(cfg.fpn_levels)
        ])
    
    def forward(self, features):
        lats = [self.laterals[i](features[i]) for i in range(len(features))]
        pyr = [None] * len(features)
        pyr[-1] = lats[-1]
        for i in range(len(features) - 2, -1, -1):
            up = F.interpolate(pyr[i+1], size=lats[i].shape[-2:], mode="bilinear", align_corners=False)
            pyr[i] = lats[i] + up
        return [self.smooth[i](pyr[i]) for i in range(len(features))]


# ──────────────────────────────────────────────────────────────────────────────
# 5. WOUND-AWARE CROSS-SCALE ATTENTION (WA-CSA)
# ──────────────────────────────────────────────────────────────────────────────
class WoundAwareCrossScaleAttention(nn.Module):
    """Bidirectional cross-attention + wound gate + tanh alpha residual."""
    
    def __init__(self, cfg: ModelConfig, level_idx: int = 0):
        super().__init__()
        dim = cfg.fpn_dim
        self.heads = cfg.wa_csa_heads
        self.head_dim = dim // self.heads
        self.scale = self.head_dim ** -0.5
        self.use_sdpa = (level_idx > 0)
        self.dropout_p = cfg.wa_csa_dropout
        
        self.q_f = nn.Linear(dim, dim, bias=False)
        self.k_c = nn.Linear(dim, dim, bias=False)
        self.v_c = nn.Linear(dim, dim, bias=False)
        self.q_c = nn.Linear(dim, dim, bias=False)
        self.k_f = nn.Linear(dim, dim, bias=False)
        self.v_f = nn.Linear(dim, dim, bias=False)
        self.out_f = nn.Linear(dim, dim, bias=False)
        self.out_c = nn.Linear(dim, dim, bias=False)
        
        self.gate_f = nn.Sequential(nn.Conv2d(dim, dim//4, 1), nn.GELU(), nn.Conv2d(dim//4, 1, 1), nn.Sigmoid())
        self.gate_c = nn.Sequential(nn.Conv2d(dim, dim//4, 1), nn.GELU(), nn.Conv2d(dim//4, 1, 1), nn.Sigmoid())
        
        self.alpha_f = nn.Parameter(torch.zeros(1))
        self.alpha_c = nn.Parameter(torch.zeros(1))
        self.norm_f = nn.LayerNorm(dim)
        self.norm_c = nn.LayerNorm(dim)
        self.drop = nn.Dropout(cfg.wa_csa_dropout)
    
    def _attn(self, q, k, v):
        q = rearrange(q, "b n (h d) -> b h n d", h=self.heads)
        k = rearrange(k, "b n (h d) -> b h n d", h=self.heads)
        v = rearrange(v, "b n (h d) -> b h n d", h=self.heads)
        if self.use_sdpa and hasattr(F, 'scaled_dot_product_attention'):
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout_p if self.training else 0.0)
        else:
            a = (q @ k.transpose(-2, -1)) * self.scale
            out = self.drop(a.softmax(-1)) @ v
        return rearrange(out, "b h n d -> b n (h d)")
    
    def forward(self, fine, coarse):
        B, C, H, W = fine.shape
        f_seq = self.norm_f(rearrange(fine, "b c h w -> b (h w) c"))
        c_seq = self.norm_c(rearrange(coarse, "b c h w -> b (h w) c"))
        
        f_up = rearrange(self.out_f(self._attn(self.q_f(f_seq), self.k_c(c_seq), self.v_c(c_seq))), "b (h w) c -> b c h w", h=H)
        c_up = rearrange(self.out_c(self._attn(self.q_c(c_seq), self.k_f(f_seq), self.v_f(f_seq))), "b (h w) c -> b c h w", h=H)
        
        return (fine + torch.tanh(self.alpha_f) * f_up * self.gate_f(fine),
                coarse + torch.tanh(self.alpha_c) * c_up * self.gate_c(coarse))


class WA_CSA_Stack(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        pairs = cfg.fpn_levels - 1
        self.layers = nn.ModuleList([
            nn.ModuleList([WoundAwareCrossScaleAttention(cfg, p) for p in range(pairs)])
            for _ in range(cfg.wa_csa_layers)
        ])
    
    def forward(self, pyr):
        for layer in self.layers:
            p = list(pyr)
            for i, wa in enumerate(layer):
                p[i], p[i+1] = wa(p[i], p[i+1])
            pyr = p
        return pyr


# ──────────────────────────────────────────────────────────────────────────────
# 6. CLASSIFICATION DECODER (MoE)
# ──────────────────────────────────────────────────────────────────────────────
class Expert(nn.Module):
    def __init__(self, d_in, d_hid, d_out):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_hid), nn.GELU(), nn.Dropout(0.1), nn.Linear(d_hid, d_out))
    def forward(self, x): return self.net(x)

class TopKRouter(nn.Module):
    def __init__(self, d_in, n_exp, top_k=2):
        super().__init__()
        self.n_exp, self.top_k = n_exp, top_k
        self.gate = nn.Linear(d_in, n_exp, bias=False)
        self.aux_loss = 0.0
    def forward(self, x):
        probs = F.softmax(self.gate(x), -1)
        w, idx = torch.topk(probs, self.top_k, -1)
        w = w / w.sum(-1, keepdim=True)
        self.aux_loss = F.mse_loss(probs.mean(0), torch.ones(self.n_exp, device=x.device) / self.n_exp)
        return w, idx

class ClassificationDecoder(nn.Module):
    """Multi-scale pool → MoE → logits + wound_embed (WTCS bridge)."""
    def __init__(self, cfg):
        super().__init__()
        F = cfg.fpn_dim
        self.cfg = cfg
        self.pools = nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(1)) for _ in range(cfg.fpn_levels)])
        self.scale_attn = nn.Sequential(nn.Linear(cfg.fpn_levels, cfg.fpn_levels), nn.Softmax(-1))
        self.pre = nn.Sequential(nn.Linear(F, F), nn.LayerNorm(F), nn.GELU())
        self.experts = nn.ModuleList([Expert(F, F*2, cfg.num_classes) for _ in range(cfg.num_experts)])
        self.router = TopKRouter(F, cfg.num_experts, cfg.top_k_experts)
        self.embed_head = nn.Sequential(nn.Linear(F, cfg.cls_embed_dim), nn.LayerNorm(cfg.cls_embed_dim), nn.GELU())
    
    def forward(self, pyr):
        B = pyr[0].shape[0]
        pooled = torch.stack([self.pools[i](pyr[i]) for i in range(self.cfg.fpn_levels)], 1)
        sw = self.scale_attn(torch.ones(B, self.cfg.fpn_levels, device=pooled.device))
        fused = self.pre((pooled * sw.unsqueeze(-1)).sum(1))
        w, idx = self.router(fused)
        logits = torch.zeros(B, self.cfg.num_classes, device=fused.device)
        for k in range(self.cfg.top_k_experts):
            for e in range(self.cfg.num_experts):
                mask = (idx[:, k] == e)
                if mask.any():
                    logits[mask] += w[mask, k:k+1] * self.experts[e](fused[mask])
        return logits, self.embed_head(fused)


# ──────────────────────────────────────────────────────────────────────────────
# 7. P-scSE MODULE (from FUSegNet — Dhar et al., 2024)
# ──────────────────────────────────────────────────────────────────────────────
class ChannelSE(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 4)
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(1),
                                nn.Linear(ch, mid, bias=False), nn.ReLU(True),
                                nn.Linear(mid, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).view(x.shape[0], -1, 1, 1)

class SpatialSE(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, 1, 1, bias=False)
    def forward(self, x):
        return x * torch.sigmoid(self.conv(x))

class ParallelScSE(nn.Module):
    """Full: additive(cSE+sSE) + maxout(cSE,sSE). Shortened: additive only."""
    def __init__(self, ch, r=16, shortened=False):
        super().__init__()
        self.shortened = shortened
        self.cse_a, self.sse_a = ChannelSE(ch, r), SpatialSE(ch)
        if not shortened:
            self.cse_m, self.sse_m = ChannelSE(ch, r), SpatialSE(ch)
    def forward(self, x):
        add = self.cse_a(x) + self.sse_a(x)
        if self.shortened:
            return add
        return add + torch.max(self.cse_m(x), self.sse_m(x))


# ──────────────────────────────────────────────────────────────────────────────
# 8. FiLM CONDITIONER (WTCS bridge)
# ──────────────────────────────────────────────────────────────────────────────
class FiLMConditioner(nn.Module):
    def __init__(self, embed_dim, feat_dim):
        super().__init__()
        self.gamma = nn.Sequential(nn.Linear(embed_dim, feat_dim), nn.Sigmoid())
        self.beta = nn.Linear(embed_dim, feat_dim)
    def forward(self, feat, embed):
        g = self.gamma(embed).unsqueeze(-1).unsqueeze(-1) + 1.0
        b = self.beta(embed).unsqueeze(-1).unsqueeze(-1)
        return g * feat + b


# ──────────────────────────────────────────────────────────────────────────────
# 9. SEGMENTATION DECODER (P-scSE + FiLM/WTCS)
#    FUSegNet arrangement: Conv-BN-ReLU → P-scSE (middle) → Conv-BN-ReLU
# ──────────────────────────────────────────────────────────────────────────────
class PscSEDecoderStage(nn.Module):
    """Conv → P-scSE (middle) → Conv. Top stage uses shortened P-scSE."""
    def __init__(self, in_ch, out_ch, r=16, shortened=False):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.pscse = ParallelScSE(out_ch, r, shortened)
        self.conv2 = nn.Sequential(nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x):
        return self.conv2(self.pscse(self.conv1(x)))

class SegmentationDecoder(nn.Module):
    """Progressive decode P4→P1 with FiLM + P-scSE at each stage."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        F = cfg.fpn_dim
        self.films = nn.ModuleList([FiLMConditioner(cfg.cls_embed_dim, F) for _ in range(cfg.fpn_levels)])
        self.stages = nn.ModuleList([
            PscSEDecoderStage(F*2, F, cfg.pscse_reduction, shortened=(i == cfg.fpn_levels - 2))
            for i in range(cfg.fpn_levels - 1)
        ])
        self.head = nn.Sequential(
            nn.Conv2d(F, F//2, 3, padding=1, bias=False), nn.BatchNorm2d(F//2), nn.ReLU(True),
            nn.Conv2d(F//2, F//4, 3, padding=1, bias=False), nn.BatchNorm2d(F//4), nn.ReLU(True),
            nn.Conv2d(F//4, cfg.seg_out_channels, 1),
        )
    
    def forward(self, pyr, wound_embed, target_size=None):
        if target_size is None: target_size = self.cfg.img_size
        cond = [self.films[i](pyr[i], wound_embed) for i in range(self.cfg.fpn_levels)]
        x = cond[-1]
        for s in range(self.cfg.fpn_levels - 1):
            skip = cond[self.cfg.fpn_levels - 2 - s]
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = self.stages[s](torch.cat([x, skip], 1))
        x = F.interpolate(x, size=target_size, mode="bilinear", align_corners=False)
        return self.head(x)


# ──────────────────────────────────────────────────────────────────────────────
# 10. DETECTION DECODER (Anchor-Free)
# ──────────────────────────────────────────────────────────────────────────────
class DetectionDecoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        F = cfg.fpn_dim
        self.level_w = nn.Parameter(torch.ones(cfg.fpn_levels) / cfg.fpn_levels)
        self.shared = nn.Sequential(
            nn.Conv2d(F, F, 3, padding=1, bias=False), nn.GroupNorm(32, F), nn.GELU(),
            nn.Conv2d(F, F, 3, padding=1, bias=False), nn.GroupNorm(32, F), nn.GELU(),
        )
        self.obj = nn.Conv2d(F, 1, 1)
        self.bbox = nn.Sequential(nn.Conv2d(F, F//2, 3, padding=1), nn.GELU(), nn.Conv2d(F//2, 4, 1), nn.Sigmoid())
        self.cls = nn.Conv2d(F, cfg.num_classes, 1)
    
    def forward(self, pyr):
        w = F.softmax(self.level_w, 0)
        tgt = pyr[0].shape[-2:]
        fused = sum(
            w[i] * (F.interpolate(p, tgt, mode="bilinear", align_corners=False) if p.shape[-2:] != tgt else p)
            for i, p in enumerate(pyr)
        )
        feat = self.shared(fused)
        return {"objectness": self.obj(feat), "bbox": self.bbox(feat), "det_cls": self.cls(feat)}


# ──────────────────────────────────────────────────────────────────────────────
# 11. WILLIE MODEL
# ──────────────────────────────────────────────────────────────────────────────
class WILLIEModel(nn.Module):
    """
    DINOv2 → FPN → WA-CSA → [CLS(MoE) + SEG(P-scSE+FiLM) + DET]
    WTCS bridge: cls wound_embed → FiLM conditioning on seg decoder.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = DINOv2MultiScale(cfg)
        self.fpn = FeaturePyramidNeck(cfg)
        self.wa_csa = WA_CSA_Stack(cfg)
        self.cls_decoder = ClassificationDecoder(cfg)
        self.seg_decoder = SegmentationDecoder(cfg)
        self.det_decoder = DetectionDecoder(cfg)
    
    def unfreeze_backbone(self):
        self.encoder.set_frozen(False)
        print("  🔓 Backbone unfrozen")
    
    def forward(self, x, target_seg_size=None):
        pyr = self.wa_csa(self.fpn(self.encoder(x)))
        logits, embed = self.cls_decoder(pyr)
        seg = self.seg_decoder(pyr, embed, target_seg_size)
        det = self.det_decoder(pyr)
        return {"cls_logits": logits, "wound_embed": embed, "seg_mask": seg,
                "det_objectness": det["objectness"], "det_bbox": det["bbox"], "det_cls": det["det_cls"]}
    
    def get_router_aux_loss(self):
        return self.cls_decoder.router.aux_loss


# ──────────────────────────────────────────────────────────────────────────────
# 12. CHECKPOINT MANAGER
# ──────────────────────────────────────────────────────────────────────────────
class CheckpointManager:
    def __init__(self, save_dir, model_name="woundshot"):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.model_name = model_name
    
    def save(self, model, optimizer=None, scheduler=None, epoch=0, fold=0, metrics=None, tag="latest"):
        ckpt = {"model_state_dict": model.state_dict(), "epoch": epoch, "fold": fold,
                "metrics": metrics or {}, "config": model.cfg.__dict__ if hasattr(model, 'cfg') else {}}
        if optimizer: ckpt["optimizer_state_dict"] = optimizer.state_dict()
        if scheduler: ckpt["scheduler_state_dict"] = scheduler.state_dict()
        path = self.save_dir / f"{self.model_name}_fold{fold}_{tag}.pt"
        tmp = path.with_suffix(".tmp")
        torch.save(ckpt, tmp); tmp.rename(path)
        return path
    
    def load(self, model, fold=0, tag="best", optimizer=None, scheduler=None, device=None):
        path = self.save_dir / f"{self.model_name}_fold{fold}_{tag}.pt"
        if not path.exists():
            print(f"  ⚠️  No checkpoint at {path}"); return None
        ckpt = torch.load(path, map_location=device or DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        if optimizer and "optimizer_state_dict" in ckpt: optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if scheduler and "scheduler_state_dict" in ckpt: scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        print(f"  ✅ Loaded: {path.name} (epoch {ckpt.get('epoch', '?')})")
        return ckpt.get("metrics", {})


# ──────────────────────────────────────────────────────────────────────────────
# 13. BUILD + VERIFY MINI
# ──────────────────────────────────────────────────────────────────────────────
def build_and_verify(variant="MINI"):
    cfg = get_model_config(variant)
    print(f"\n{'─'*80}")
    print(f"  🏗️  Building WILLIE-{variant} (P-scSE Seg)")
    print(f"{'─'*80}")
    
    model = WILLIEModel(cfg).to(DEVICE)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\n  📊 Params: {total/1e6:.2f}M total, {trainable/1e6:.2f}M trainable, {(total-trainable)/1e6:.2f}M frozen")
    
    for name, mod in [("encoder", model.encoder), ("fpn", model.fpn), ("wa_csa", model.wa_csa),
                       ("cls_decoder", model.cls_decoder), ("seg_decoder(P-scSE)", model.seg_decoder),
                       ("det_decoder", model.det_decoder)]:
        n = sum(p.numel() for p in mod.parameters()) / 1e6
        print(f"    {name:30s}: {n:8.2f}M ({n*1e6/total*100:5.1f}%)")
    
    # Forward test
    B = 2
    dummy = torch.randn(B, 3, cfg.img_size, cfg.img_size, device=DEVICE)
    with torch.no_grad():
        out = model(dummy, target_seg_size=cfg.img_size)
    print(f"\n  🧪 Forward:")
    for k, v in out.items():
        if isinstance(v, torch.Tensor): print(f"    {k}: {v.shape}")
    
    # Backward test
    model.train()
    out = model(dummy, target_seg_size=cfg.img_size)
    loss = (F.cross_entropy(out["cls_logits"], torch.randint(0, cfg.num_classes, (B,), device=DEVICE))
            + F.binary_cross_entropy_with_logits(out["seg_mask"], torch.rand(B, 1, cfg.img_size, cfg.img_size, device=DEVICE))
            + out["det_objectness"].mean() + model.get_router_aux_loss() * 0.01)
    loss.backward()
    grads = sum(1 for p in model.parameters() if p.grad is not None)
    total_p = sum(1 for p in model.parameters())
    print(f"  🔙 Backward: {grads}/{total_p} params have grads ✅")
    
    if torch.cuda.is_available():
        peak = torch.cuda.max_memory_allocated() / 1e9
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  💾 Memory: {peak:.1f}GB / {total_memory:.1f}GB ({total_memory-peak:.1f}GB free)")
        torch.cuda.reset_peak_memory_stats()
    
    print(f"\n{'='*80}")
    print(f"  ✅ Cell 1 COMPLETE — WoundShot-{variant} ({total/1e6:.1f}M) + All Manifests Loaded")
    print(f"{'='*80}")
    return model, cfg

model, cfg = build_and_verify("MINI")
ckpt_mgr = CheckpointManager(CKPT_DIR / "mini", model_name="woundshot_mini")

  09_WILLIE_FUSEG_CSD — Cell 1
  2026-03-23 14:31:27
  GPU: Tesla V100-PCIE-32GB (34.1 GB)
  PyTorch: 2.10.0+cu128

────────────────────────────────────────────────────────────────────────────────
  📂 Loading CSV Manifests
────────────────────────────────────────────────────────────────────────────────
  ✅ cls_train   :   918 rows  ← cls_train.csv
  ✅ cls_val     :   162 rows  ← cls_val.csv
  ✅ cls_test    :   234 rows  ← cls_test.csv
  ✅ seg_train   :   610 rows  ← ws_seg_manifest_fuseg_train.csv
  ✅ seg_val     :   400 rows  ← ws_seg_manifest_fuseg_val.csv
  ✅ det_train   :   853 rows  ← ws_det_manifest_yolo_train.csv
  ✅ det_val     :   367 rows  ← ws_det_manifest_yolo_val.csv

  Classification columns: img=image_path, class=unified_class, label=unified_label
  Train distribution:
    diabetic       :  187
    no_wound       :  128
    pressure       :  229
    surgical       :  104
    venous         :  270

  Segmentation columns: img=img, mask=mask
  Detection columns: img=img, b

Using cache found in <HOME>/.cache/torch/hub/facebookresearch_dinov2_main



  📊 Params: 34.32M total, 12.26M trainable, 22.06M frozen
    encoder                       :    22.06M ( 64.3%)
    fpn                           :     2.76M (  8.0%)
    wa_csa                        :     1.68M (  4.9%)
    cls_decoder                   :     0.37M (  1.1%)
    seg_decoder(P-scSE)           :     5.99M ( 17.4%)
    det_decoder                   :     1.48M (  4.3%)

  🧪 Forward:
    cls_logits: torch.Size([2, 5])
    wound_embed: torch.Size([2, 128])
    seg_mask: torch.Size([2, 1, 518, 518])
    det_objectness: torch.Size([2, 1, 37, 37])
    det_bbox: torch.Size([2, 4, 37, 37])
    det_cls: torch.Size([2, 5, 37, 37])
  🔙 Backward: 175/356 params have grads ✅
  💾 Memory: 4.3GB / 34.1GB (29.8GB free)

  ✅ Cell 1 COMPLETE — WILLIE-MINI (34.3M) + All Manifests Loaded


In [2]:
"""
================================================================================
  09_WILLIE_FUSEG_CSD.ipynb — Cell 2
  DataLoaders + 5-Fold Splits + Multi-Task Loss + Combined Metric
  
  Uses MANIFESTS dict from Cell 1.
  Column mapping:
    cls: image_path, unified_label, unified_class
    seg: img, mask
    det: img, label
================================================================================
"""

import random, cv2, ast
from collections import Counter
from PIL import Image
from sklearn.model_selection import StratifiedKFold

import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

print(f"\n{'='*80}")
print(f"  Cell 2: DataLoaders + 5-Fold Splits + Loss")
print(f"{'='*80}")

# ──────────────────────────────────────────────────────────────────────────────
# 0. DELETE STALE SPLITS FROM PREVIOUS RUNS
# ──────────────────────────────────────────────────────────────────────────────
SPLITS_FILE = CKPT_DIR / "5fold_splits_v2.pt"  # v2 to avoid stale cache
MANIFESTS_CACHE = CKPT_DIR / "data_manifests.pt"
IMG_SIZE = cfg.img_size  # 518

# Remove stale files from previous broken runs
for stale in [CKPT_DIR / "5fold_splits.pt", MANIFESTS_CACHE]:
    if stale.exists():
        stale.unlink()
        print(f"  🗑️  Removed stale: {stale.name}")

# ──────────────────────────────────────────────────────────────────────────────
# 1. MERGE TRAIN+VAL FOR 5-FOLD CV
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  📦 Building 5-Fold Cross-Validation Splits")
print(f"{'─'*80}")

cls_all = pd.concat([MANIFESTS["cls_train"], MANIFESTS["cls_val"]], ignore_index=True)
cls_test = MANIFESTS["cls_test"].copy()
seg_all = pd.concat([MANIFESTS["seg_train"], MANIFESTS["seg_val"]], ignore_index=True)
det_all = pd.concat([MANIFESTS["det_train"], MANIFESTS["det_val"]], ignore_index=True)

print(f"  Classification: {len(cls_all)} train+val, {len(cls_test)} test")
print(f"  Segmentation:   {len(seg_all)} total")
print(f"  Detection:       {len(det_all)} total")

N_FOLDS = 5

def create_5fold_splits():
    if SPLITS_FILE.exists():
        print(f"\n  ✅ Loading saved splits from {SPLITS_FILE.name}")
        return torch.load(SPLITS_FILE, weights_only=False)
    
    print(f"\n  🔀 Creating {N_FOLDS}-fold stratified splits...")
    
    # Stratified for classification
    cls_labels = cls_all["unified_label"].values
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    cls_folds = list(skf.split(np.arange(len(cls_all)), cls_labels))
    
    # Proportional for segmentation
    seg_idx = np.arange(len(seg_all))
    np.random.RandomState(SEED).shuffle(seg_idx)
    seg_size = len(seg_idx) // N_FOLDS
    
    # Proportional for detection
    det_idx = np.arange(len(det_all))
    np.random.RandomState(SEED + 1).shuffle(det_idx)
    det_size = len(det_idx) // N_FOLDS
    
    folds = {}
    for fold in range(N_FOLDS):
        cls_tr, cls_va = cls_folds[fold]
        
        s0 = fold * seg_size
        s1 = s0 + seg_size if fold < N_FOLDS - 1 else len(seg_idx)
        seg_va = seg_idx[s0:s1]
        seg_tr = np.concatenate([seg_idx[:s0], seg_idx[s1:]])
        
        d0 = fold * det_size
        d1 = d0 + det_size if fold < N_FOLDS - 1 else len(det_idx)
        det_va = det_idx[d0:d1]
        det_tr = np.concatenate([det_idx[:d0], det_idx[d1:]])
        
        folds[fold] = {
            "cls_tr": cls_tr.tolist(), "cls_va": cls_va.tolist(),
            "seg_tr": seg_tr.tolist(), "seg_va": seg_va.tolist(),
            "det_tr": det_tr.tolist(), "det_va": det_va.tolist(),
        }
        print(f"  Fold {fold}: cls={len(cls_tr)}/{len(cls_va)}, "
              f"seg={len(seg_tr)}/{len(seg_va)}, det={len(det_tr)}/{len(det_va)}")
    
    data = {"n_folds": N_FOLDS, "folds": folds}
    torch.save(data, SPLITS_FILE)
    print(f"  💾 Saved to {SPLITS_FILE.name}")
    return data

splits = create_5fold_splits()

# ──────────────────────────────────────────────────────────────────────────────
# 2. AUGMENTATIONS
# ──────────────────────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transforms(sz=IMG_SIZE):
    return A.Compose([
        A.Resize(sz, sz),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.5,
                           border_mode=cv2.BORDER_CONSTANT, value=0),
        A.OneOf([A.ElasticTransform(alpha=30, sigma=5, p=0.3),
                 A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3)], p=0.25),
        A.OneOf([A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
                 A.GaussianBlur(blur_limit=(3, 5), p=0.3)], p=0.25),
        A.OneOf([A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
                 A.HueSaturationValue(10, 20, 15, p=0.4),
                 A.CLAHE(clip_limit=2.0, p=0.3)], p=0.4),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transforms(sz=IMG_SIZE):
    return A.Compose([
        A.Resize(sz, sz),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_tta_transforms(sz=IMG_SIZE):
    n = [A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return [
        A.Compose([A.Resize(sz, sz)] + n),
        A.Compose([A.Resize(sz, sz), A.HorizontalFlip(p=1.0)] + n),
        A.Compose([A.Resize(sz, sz), A.VerticalFlip(p=1.0)] + n),
        A.Compose([A.Resize(sz, sz), A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)] + n),
        A.Compose([A.Resize(sz, sz), A.RandomRotate90(p=1.0)] + n),
    ]

# ──────────────────────────────────────────────────────────────────────────────
# 3. BBOX HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def parse_yolo_bbox(bbox_str):
    """'[[cls,cx,cy,w,h]]' → (N,4) as [x1,y1,x2,y2] normalized."""
    if pd.isna(bbox_str) or str(bbox_str).strip() in ("", "[]", "nan"):
        return np.zeros((0, 4), dtype=np.float32)
    try:
        raw = ast.literal_eval(str(bbox_str))
    except:
        return np.zeros((0, 4), dtype=np.float32)
    if not raw: return np.zeros((0, 4), dtype=np.float32)
    out = []
    for bb in raw:
        if len(bb) >= 5: _, cx, cy, w, h = bb[:5]
        elif len(bb) == 4: cx, cy, w, h = bb
        else: continue
        out.append([max(0, cx-w/2), max(0, cy-h/2), min(1, cx+w/2), min(1, cy+h/2)])
    return np.array(out, dtype=np.float32) if out else np.zeros((0, 4), dtype=np.float32)

def mask_to_bboxes(mask_np):
    """Connected components → [x1,y1,x2,y2] normalized."""
    if mask_np.max() == 0: return np.zeros((0, 4), dtype=np.float32)
    binary = (mask_np > 0.5).astype(np.uint8)
    n_lab, _, stats, _ = cv2.connectedComponentsWithStats(binary, 8)
    H, W = mask_np.shape
    out = []
    for i in range(1, n_lab):
        x, y, w, h, a = stats[i]
        if a < 50: continue
        out.append([x/W, y/H, (x+w)/W, (y+h)/H])
    return np.array(out, dtype=np.float32) if out else np.zeros((0, 4), dtype=np.float32)

# ──────────────────────────────────────────────────────────────────────────────
# 4. MULTI-TASK DATASET
# ──────────────────────────────────────────────────────────────────────────────
class WoundMultiTaskDataset(Dataset):
    def __init__(self, cls_df, seg_df, det_df, transform=None, img_size=518):
        super().__init__()
        self.transform = transform
        self.img_size = img_size
        self.samples = []
        seen = set()
        
        # Cls samples
        if len(cls_df) > 0:
            for _, row in cls_df.iterrows():
                p = str(row["image_path"])
                self.samples.append({"path": p, "cls_label": int(row["unified_label"]),
                                     "has_mask": False, "mask_path": None, "bbox_str": None})
                seen.add(os.path.normpath(p))
        
        # Seg lookup
        seg_lookup = {}
        if len(seg_df) > 0:
            for _, row in seg_df.iterrows():
                seg_lookup[os.path.normpath(str(row["img"]))] = str(row["mask"])
        
        # Det samples
        if len(det_df) > 0:
            det_img_col = "img" if "img" in det_df.columns else "image_path"
            det_bbox_col = "label" if "label" in det_df.columns else "bbox_yolo"
            
            for _, row in det_df.iterrows():
                p = str(row[det_img_col])
                np_ = os.path.normpath(p)
                if np_ in seen: continue
                
                mp = seg_lookup.get(np_)
                has_mp = mp is not None
                
                bbox_str = str(row[det_bbox_col]) if det_bbox_col in det_df.columns else "[]"
                
                self.samples.append({"path": p, "cls_label": -1, "has_mask": has_mp,
                                     "mask_path": mp, "bbox_str": bbox_str})
                seen.add(np_)
        
        # Remaining seg-only
        for np_, mp in seg_lookup.items():
            if np_ not in seen:
                self.samples.append({"path": np_, "cls_label": -1, "has_mask": True,
                                     "mask_path": mp, "bbox_str": None})
                seen.add(np_)
        
        n_c = sum(1 for s in self.samples if s["cls_label"] >= 0)
        n_s = sum(1 for s in self.samples if s["has_mask"])
        print(f"    Dataset: {len(self.samples)} total ({n_c} cls, {n_s} seg)")
    
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = cv2.imread(s["path"])
        if img is None:
            img = np.array(Image.open(s["path"]).convert("RGB"))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if s["has_mask"] and s["mask_path"]:
            mask = cv2.imread(s["mask_path"], cv2.IMREAD_GRAYSCALE)
            if mask is None: mask = np.array(Image.open(s["mask_path"]).convert("L"))
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.float32)
        
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]
        
        if isinstance(mask, np.ndarray): mask = torch.from_numpy(mask)
        mask = mask.float().unsqueeze(0)
        
        mask_np = mask.squeeze(0).numpy()
        if s["has_mask"] and mask_np.max() > 0:
            bboxes = mask_to_bboxes(mask_np)
        elif s["bbox_str"]:
            bboxes = parse_yolo_bbox(s["bbox_str"])
        else:
            bboxes = np.zeros((0, 4), dtype=np.float32)
        
        return {"image": img, "cls_label": s["cls_label"], "seg_mask": mask,
                "det_bboxes": torch.from_numpy(bboxes), "has_mask": s["has_mask"]}


def collate_multitask(batch):
    images = torch.stack([b["image"] for b in batch])
    cls_labels = torch.tensor([b["cls_label"] for b in batch], dtype=torch.long)
    seg_masks = torch.stack([b["seg_mask"] for b in batch])
    has_mask = torch.tensor([b["has_mask"] for b in batch], dtype=torch.bool)
    
    max_b = max((b["det_bboxes"].shape[0] for b in batch), default=0)
    max_b = max(max_b, 1)
    det_bboxes = torch.zeros(len(batch), max_b, 4)
    det_valid = torch.zeros(len(batch), max_b, dtype=torch.bool)
    for i, b in enumerate(batch):
        n = b["det_bboxes"].shape[0]
        if n > 0:
            det_bboxes[i, :n] = b["det_bboxes"]
            det_valid[i, :n] = True
    
    return {"image": images, "cls_label": cls_labels, "seg_mask": seg_masks,
            "det_bboxes": det_bboxes, "det_valid": det_valid, "has_mask": has_mask}

# ──────────────────────────────────────────────────────────────────────────────
# 5. MULTI-TASK LOSS
# ──────────────────────────────────────────────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, pred, target):
        p = torch.sigmoid(pred).flatten(1)
        t = target.flatten(1)
        inter = (p * t).sum(1)
        return 1.0 - ((2*inter + self.smooth) / (p.sum(1) + t.sum(1) + self.smooth)).mean()

class MultiTaskLoss(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.cls_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.seg_bce = nn.BCEWithLogitsLoss()
        self.seg_dice = DiceLoss()
        self.det_obj = nn.BCEWithLogitsLoss()
        self.log_var_cls = nn.Parameter(torch.zeros(1))
        self.log_var_seg = nn.Parameter(torch.zeros(1))
        self.log_var_det = nn.Parameter(torch.zeros(1))
    
    def forward(self, pred, batch, aux_loss=None):
        losses = {}
        dev = pred["cls_logits"].device
        
        labels = batch["cls_label"].to(dev)
        valid = labels >= 0
        losses["cls"] = self.cls_fn(pred["cls_logits"][valid], labels[valid]) if valid.any() else torch.tensor(0.0, device=dev)
        
        hm = batch["has_mask"].to(dev)
        if hm.any():
            sp, st = pred["seg_mask"][hm], batch["seg_mask"][hm].to(dev)
            losses["seg"] = self.seg_bce(sp, st) + self.seg_dice(sp, st)
            do = pred["det_objectness"][hm]
            ot = F.interpolate(st, do.shape[-2:], mode="bilinear", align_corners=False)
            losses["det"] = self.det_obj(do, (ot > 0.3).float())
        else:
            losses["seg"] = torch.tensor(0.0, device=dev)
            losses["det"] = torch.tensor(0.0, device=dev)
        
        losses["aux"] = aux_loss if aux_loss is not None else torch.tensor(0.0, device=dev)
        
        wc, ws, wd = torch.exp(-self.log_var_cls), torch.exp(-self.log_var_seg), torch.exp(-self.log_var_det)
        losses["total"] = (wc*losses["cls"] + self.log_var_cls
                         + ws*losses["seg"] + self.log_var_seg
                         + wd*losses["det"] + self.log_var_det
                         + 0.01*losses["aux"])
        losses["w_cls"], losses["w_seg"], losses["w_det"] = wc.item(), ws.item(), wd.item()
        return losses

# ──────────────────────────────────────────────────────────────────────────────
# 6. COMBINED METRIC
# ──────────────────────────────────────────────────────────────────────────────
def compute_combined_metric(cls_acc, seg_dice, det_ap50):
    return {
        "cls_acc": cls_acc, "seg_dice": seg_dice, "det_ap50": det_ap50,
        "combined_equal": (cls_acc + seg_dice + det_ap50) / 3,
        "combined_weighted": 0.4*cls_acc + 0.4*seg_dice + 0.2*det_ap50,
        "combined_cls_seg": (cls_acc + seg_dice) / 2,
        "min_task": min(cls_acc, seg_dice, det_ap50),
    }

# ──────────────────────────────────────────────────────────────────────────────
# 7. DATALOADER FACTORY
# ──────────────────────────────────────────────────────────────────────────────
BATCH_SIZE = 4
NUM_WORKERS = 4

def get_fold_dataloaders(fold, batch_size=BATCH_SIZE):
    f = splits["folds"][fold]
    
    cls_tr = cls_all.iloc[f["cls_tr"]].reset_index(drop=True)
    cls_va = cls_all.iloc[f["cls_va"]].reset_index(drop=True)
    seg_tr = seg_all.iloc[f["seg_tr"]].reset_index(drop=True)
    seg_va = seg_all.iloc[f["seg_va"]].reset_index(drop=True)
    det_tr = det_all.iloc[f["det_tr"]].reset_index(drop=True)
    det_va = det_all.iloc[f["det_va"]].reset_index(drop=True)
    
    print(f"\n  📦 Fold {fold}:")
    train_ds = WoundMultiTaskDataset(cls_tr, seg_tr, det_tr, get_train_transforms(), IMG_SIZE)
    val_ds   = WoundMultiTaskDataset(cls_va, seg_va, det_va, get_val_transforms(), IMG_SIZE)
    
    # Balanced sampling
    labels = [s["cls_label"] for s in train_ds.samples if s["cls_label"] >= 0]
    if labels:
        counts = Counter(labels)
        tot = len(labels)
        cw = {c: tot/n for c, n in counts.items()}
        wts = [cw.get(s["cls_label"], 1.0) for s in train_ds.samples]
        sampler = WeightedRandomSampler(wts, len(wts), replacement=True)
    else:
        sampler = None
    
    tl = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, collate_fn=collate_multitask)
    vl = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_multitask)
    print(f"    Train: {len(train_ds)} → {len(tl)} batches | Val: {len(val_ds)} → {len(vl)} batches")
    return tl, vl

def get_test_dataloader(batch_size=BATCH_SIZE):
    empty = pd.DataFrame()
    print(f"\n  🧪 Test set:")
    ds = WoundMultiTaskDataset(cls_test, empty, empty, get_val_transforms(), IMG_SIZE)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_multitask)
    print(f"    Test: {len(ds)} → {len(dl)} batches")
    return dl

# ──────────────────────────────────────────────────────────────────────────────
# 8. VERIFY
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  🔍 Verification: Fold 0")
print(f"{'─'*80}")

train_loader_v, val_loader_v = get_fold_dataloaders(0)
batch = next(iter(train_loader_v))

print(f"\n  Batch shapes:")
print(f"    image:      {batch['image'].shape}")
print(f"    cls_label:  {batch['cls_label'].shape} → {batch['cls_label'].tolist()}")
print(f"    seg_mask:   {batch['seg_mask'].shape}")
print(f"    det_bboxes: {batch['det_bboxes'].shape}")
print(f"    has_mask:   {batch['has_mask'].tolist()}")

print(f"\n  Testing MultiTaskLoss...")
criterion = MultiTaskLoss(NUM_CLASSES).to(DEVICE)
batch_gpu = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
model.eval()
with torch.no_grad():
    pred = model(batch_gpu["image"], target_seg_size=IMG_SIZE)
    losses = criterion(pred, batch_gpu, model.get_router_aux_loss())

print(f"    cls={losses['cls'].item():.4f}  seg={losses['seg'].item():.4f}  "
      f"det={losses['det'].item():.4f}  total={losses['total'].item():.4f}")
print(f"    weights: cls={losses['w_cls']:.3f} seg={losses['w_seg']:.3f} det={losses['w_det']:.3f}")

m = compute_combined_metric(90.0, 85.0, 60.0)
print(f"\n  📊 Combined Metric (90/85/60): equal={m['combined_equal']:.1f}, weighted={m['combined_weighted']:.1f}")

del train_loader_v, val_loader_v, batch, batch_gpu
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*80}")
print(f"  ✅ Cell 2 COMPLETE — Splits + Loaders + Loss + Metric")
print(f"  Splits: {SPLITS_FILE}")
print(f"  Ready for Cell 3 (Train MINI)")
print(f"{'='*80}")


  Cell 2: DataLoaders + 5-Fold Splits + Loss

────────────────────────────────────────────────────────────────────────────────
  📦 Building 5-Fold Cross-Validation Splits
────────────────────────────────────────────────────────────────────────────────
  Classification: 1080 train+val, 234 test
  Segmentation:   1010 total
  Detection:       1220 total

  ✅ Loading saved splits from 5fold_splits_v2.pt

────────────────────────────────────────────────────────────────────────────────
  🔍 Verification: Fold 0
────────────────────────────────────────────────────────────────────────────────

  📦 Fold 0:
    Dataset: 2648 total (864 cls, 808 seg)
    Dataset: 662 total (216 cls, 202 seg)
    Train: 2648 → 662 batches | Val: 662 → 166 batches

  Batch shapes:
    image:      torch.Size([4, 3, 518, 518])
    cls_label:  torch.Size([4]) → [-1, 2, -1, 3]
    seg_mask:   torch.Size([4, 1, 518, 518])
    det_bboxes: torch.Size([4, 3, 4])
    has_mask:   [True, False, False, False]

  Testing Multi

In [3]:
"""
================================================================================
  09_WILLIE_FUSEG_CSD.ipynb — Cell 3 (v3 — FIXED DETECTION METRIC)
  Train WILLIE-MINI (5-Fold CV)
  
  Target: 90%+ cls, 90%+ seg Dice, 90%+ det AP@0.5 → 95% combined
  
  FIXES from v2:
    ✓ Detection: real IoU-based AP@0.5 from seg→bbox connected components
    ✓ Seg target resize: _match_size handles 518→512
    ✓ Checkpoints: always saves, resumes from crash
================================================================================
"""

import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from tqdm.notebook import tqdm

print(f"\n{'='*80}")
print(f"  Cell 3: Train WILLIE-MINI (5-Fold CV)")
print(f"  Target: 90%+ per task, 95% combined")
print(f"{'='*80}")

# ──────────────────────────────────────────────────────────────────────────────
# 0. TRAINING CONFIG
# ──────────────────────────────────────────────────────────────────────────────
TRAIN_CFG = {
    "variant": "MINI",
    "n_folds": 5,
    "epochs": 50,
    "freeze_epochs": 5,
    "lr_head": 1e-4,
    "lr_backbone": 1e-5,
    "weight_decay": 1e-4,
    "patience": 12,
    "grad_clip": 1.0,
    "batch_size": BATCH_SIZE,
    "accumulation_steps": 2,
    "seg_size": 512,
}

for k, v in TRAIN_CFG.items():
    print(f"    {k}: {v}")

# ── ATOMIC RESUME: Load progress from previous runs ──
PROGRESS_FILE = CKPT_DIR / "mini_training_progress.pt"

def _atomic_save(data, path):
    """Write to .tmp then rename — survives crashes/disconnects."""
    tmp = Path(str(path) + ".tmp")
    torch.save(data, tmp)
    tmp.rename(path)

def _load_progress():
    if PROGRESS_FILE.exists():
        p = torch.load(PROGRESS_FILE, map_location="cpu", weights_only=False)
        print(f"  ↩️  Resuming: {len(p.get('completed_folds', {}))} folds already done")
        return p
    return {"completed_folds": {}, "completed_histories": {}, "start_time": time.time()}

training_progress = _load_progress()

# ──────────────────────────────────────────────────────────────────────────────
# 1. SIZE-SAFE MULTI-TASK LOSS
# ──────────────────────────────────────────────────────────────────────────────
class MultiTaskLossSafe(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.cls_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.seg_bce = nn.BCEWithLogitsLoss()
        self.seg_dice = DiceLoss()
        self.det_obj = nn.BCEWithLogitsLoss()
        self.log_var_cls = nn.Parameter(torch.zeros(1))
        self.log_var_seg = nn.Parameter(torch.zeros(1))
        self.log_var_det = nn.Parameter(torch.zeros(1))
    
    @staticmethod
    def _match_size(target, pred):
        if target.shape[-2:] != pred.shape[-2:]:
            return F.interpolate(target, pred.shape[-2:], mode="bilinear", align_corners=False)
        return target
    
    def forward(self, pred, batch, aux_loss=None):
        losses = {}
        dev = pred["cls_logits"].device
        
        labels = batch["cls_label"].to(dev)
        valid = labels >= 0
        losses["cls"] = self.cls_fn(pred["cls_logits"][valid], labels[valid]) if valid.any() else torch.tensor(0.0, device=dev)
        
        hm = batch["has_mask"].to(dev)
        if hm.any():
            sp = pred["seg_mask"][hm]
            st = self._match_size(batch["seg_mask"][hm].to(dev), sp)
            losses["seg"] = self.seg_bce(sp, st) + self.seg_dice(sp, st)
            
            do = pred["det_objectness"][hm]
            ot = F.interpolate(st, do.shape[-2:], mode="bilinear", align_corners=False)
            losses["det"] = self.det_obj(do, (ot > 0.3).float())
        else:
            losses["seg"] = torch.tensor(0.0, device=dev)
            losses["det"] = torch.tensor(0.0, device=dev)
        
        losses["aux"] = aux_loss if aux_loss is not None else torch.tensor(0.0, device=dev)
        
        wc, ws, wd = torch.exp(-self.log_var_cls), torch.exp(-self.log_var_seg), torch.exp(-self.log_var_det)
        losses["total"] = (wc*losses["cls"] + self.log_var_cls
                         + ws*losses["seg"] + self.log_var_seg
                         + wd*losses["det"] + self.log_var_det
                         + 0.01*losses["aux"])
        losses["w_cls"], losses["w_seg"], losses["w_det"] = wc.item(), ws.item(), wd.item()
        return losses


# ──────────────────────────────────────────────────────────────────────────────
# 2. REAL DETECTION METRIC — IoU-based AP@0.5 from seg→bbox
# ──────────────────────────────────────────────────────────────────────────────
def compute_iou(box_a, box_b):
    """IoU between two boxes [x1,y1,x2,y2]."""
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / (union + 1e-8)


def mask_to_bboxes_eval(mask_np, min_area=50):
    """Connected components → list of [x1,y1,x2,y2] in pixel coords."""
    if mask_np.max() == 0:
        return []
    binary = (mask_np > 0.5).astype(np.uint8)
    n_lab, _, stats, _ = cv2.connectedComponentsWithStats(binary, 8)
    H, W = mask_np.shape
    boxes = []
    for i in range(1, n_lab):
        x, y, w, h, a = stats[i]
        if a < min_area:
            continue
        boxes.append([x / W, y / H, (x + w) / W, (y + h) / H])
    return boxes


def compute_ap50_single(pred_boxes, gt_boxes, iou_thresh=0.5):
    """
    AP@0.5 for a single image.
    pred_boxes: list of [x1,y1,x2,y2] (sorted by confidence, but we treat all equal)
    gt_boxes:   list of [x1,y1,x2,y2]
    """
    if len(gt_boxes) == 0 and len(pred_boxes) == 0:
        return 1.0  # true negative
    if len(gt_boxes) == 0 and len(pred_boxes) > 0:
        return 0.0  # all false positives
    if len(gt_boxes) > 0 and len(pred_boxes) == 0:
        return 0.0  # all missed
    
    matched_gt = set()
    tp, fp = 0, 0
    
    for pb in pred_boxes:
        best_iou, best_j = 0, -1
        for j, gb in enumerate(gt_boxes):
            if j in matched_gt:
                continue
            iou = compute_iou(pb, gb)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thresh and best_j >= 0:
            tp += 1
            matched_gt.add(best_j)
        else:
            fp += 1
    
    fn = len(gt_boxes) - len(matched_gt)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    # Simple AP approximation: P × R (for single-threshold)
    return precision * recall if (precision + recall) > 0 else 0.0


@torch.no_grad()
def compute_det_ap50_batch(pred_seg_logits, gt_seg_masks):
    """
    Real AP@0.5: pred seg mask → connected components → bboxes vs GT bboxes.
    Returns list of per-image AP@0.5 scores.
    """
    pred_masks = (torch.sigmoid(pred_seg_logits) > 0.5).float().cpu().numpy()
    gt_masks = gt_seg_masks.cpu().numpy()
    
    ap_scores = []
    for i in range(pred_masks.shape[0]):
        pm = pred_masks[i, 0]  # (H, W)
        gm = gt_masks[i, 0]    # (H, W)
        
        pred_boxes = mask_to_bboxes_eval(pm)
        gt_boxes = mask_to_bboxes_eval(gm)
        
        ap = compute_ap50_single(pred_boxes, gt_boxes)
        ap_scores.append(ap)
    
    return ap_scores


# ──────────────────────────────────────────────────────────────────────────────
# 3. EVALUATION (with real metrics)
# ──────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def compute_seg_dice_batch(pred_mask, gt_mask, threshold=0.5):
    pred = (torch.sigmoid(pred_mask) > threshold).float()
    dices = []
    for i in range(pred.shape[0]):
        p, g = pred[i].flatten(), gt_mask[i].flatten()
        inter = (p * g).sum()
        union = p.sum() + g.sum()
        dices.append((2 * inter / (union + 1e-8)).item() if union > 0 else (1.0 if g.sum() == 0 else 0.0))
    return dices


@torch.no_grad()
def evaluate_fold(model, val_loader, criterion, seg_size, verbose=False):
    model.eval()
    all_cls_preds, all_cls_labels = [], []
    all_dices = []
    all_det_ap = []
    total_loss, n_bat = 0.0, 0
    
    loader = tqdm(val_loader, desc="    Eval", leave=False) if verbose else val_loader
    
    for batch in loader:
        bg = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        pred = model(bg["image"], target_seg_size=seg_size)
        losses = criterion(pred, bg, model.get_router_aux_loss())
        total_loss += losses["total"].item()
        n_bat += 1
        
        # Classification
        labels = bg["cls_label"]
        valid = labels >= 0
        if valid.any():
            all_cls_preds.extend(pred["cls_logits"][valid].argmax(1).cpu().numpy())
            all_cls_labels.extend(labels[valid].cpu().numpy())
        
        # Segmentation + Detection
        hm = bg["has_mask"]
        if hm.any():
            sp = pred["seg_mask"][hm]
            st = bg["seg_mask"][hm]
            if st.shape[-2:] != sp.shape[-2:]:
                st = F.interpolate(st, sp.shape[-2:], mode="bilinear", align_corners=False)
            
            # Seg Dice
            all_dices.extend(compute_seg_dice_batch(sp, st))
            
            # Det AP@0.5 (real — seg→connected components→bbox IoU matching)
            all_det_ap.extend(compute_det_ap50_batch(sp, st))
    
    cls_acc = accuracy_score(all_cls_labels, all_cls_preds) * 100 if all_cls_labels else 0.0
    cls_f1 = f1_score(all_cls_labels, all_cls_preds, average="macro") * 100 if all_cls_labels else 0.0
    seg_dice = np.mean(all_dices) * 100 if all_dices else 0.0
    det_ap50 = np.mean(all_det_ap) * 100 if all_det_ap else 0.0
    
    cm = compute_combined_metric(cls_acc, seg_dice, det_ap50)
    return {"cls_acc": cls_acc, "cls_f1": cls_f1, "seg_dice": seg_dice,
            "seg_n": len(all_dices), "det_ap50": det_ap50, "det_n": len(all_det_ap),
            "loss": total_loss / max(n_bat, 1),
            "cls_preds": np.array(all_cls_preds), "cls_labels": np.array(all_cls_labels),
            **cm}


# ──────────────────────────────────────────────────────────────────────────────
# 4. TRAINING LOOP (ONE FOLD)
# ──────────────────────────────────────────────────────────────────────────────
def train_one_fold(fold, variant="MINI"):
    print(f"\n{'━'*80}")
    print(f"  🏋️  FOLD {fold} — WILLIE-{variant}")
    print(f"{'━'*80}")
    
    fold_cfg = get_model_config(variant)
    fold_model = WILLIEModel(fold_cfg).to(DEVICE)
    fold_criterion = MultiTaskLossSafe(NUM_CLASSES).to(DEVICE)
    fold_ckpt = CheckpointManager(CKPT_DIR / variant.lower(), f"woundshot_{variant.lower()}")
    
    train_loader, val_loader = get_fold_dataloaders(fold, TRAIN_CFG["batch_size"])
    
    backbone_params = list(fold_model.encoder.parameters())
    decoder_params = [p for n, p in fold_model.named_parameters() if "encoder" not in n]
    loss_params = list(fold_criterion.parameters())
    
    optimizer = torch.optim.AdamW([
        {"params": decoder_params, "lr": TRAIN_CFG["lr_head"]},
        {"params": loss_params, "lr": TRAIN_CFG["lr_head"]},
        {"params": backbone_params, "lr": TRAIN_CFG["lr_backbone"]},
    ], weight_decay=TRAIN_CFG["weight_decay"])
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)
    scaler = GradScaler()
    
    start_epoch, best_combined, patience_ctr = 0, 0.0, 0
    rm = fold_ckpt.load(fold_model, fold=fold, tag="latest", optimizer=optimizer, scheduler=scheduler)
    if rm:
        start_epoch = rm.get("epoch", 0) + 1
        best_combined = rm.get("best_combined", 0.0)
        patience_ctr = rm.get("patience_ctr", 0)
        if "scaler_state" in rm:
            scaler.load_state_dict(rm["scaler_state"])
        print(f"  ↩️  Resumed fold {fold} at epoch {start_epoch}, best={best_combined:.2f}")
    
    epochs = TRAIN_CFG["epochs"]
    accum = TRAIN_CFG["accumulation_steps"]
    seg_size = TRAIN_CFG["seg_size"]
    
    # Restore history from checkpoint if resuming mid-fold
    default_hist = {"train_loss": [], "val_loss": [], "cls_acc": [], "cls_f1": [],
                    "seg_dice": [], "det_ap50": [], "combined": [], "lr": [],
                    "cls_loss": [], "seg_loss": [], "det_loss": [],
                    "w_cls": [], "w_seg": [], "w_det": []}
    hist = rm.get("history", default_hist) if rm else default_hist
    
    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        
        if epoch == TRAIN_CFG["freeze_epochs"]:
            fold_model.unfreeze_backbone()
        
        # ── Train ──
        fold_model.train()
        ep_loss, ep_cls, ep_seg, ep_det = 0.0, 0.0, 0.0, 0.0
        n_steps = 0
        optimizer.zero_grad()
        last_w = {"cls": 1.0, "seg": 1.0, "det": 1.0}
        
        pbar = tqdm(train_loader, desc=f"  E{epoch:02d} Train", leave=False,
                    bar_format="{l_bar}{bar:30}{r_bar}")
        
        for step, batch in enumerate(pbar):
            bg = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            with autocast():
                pred = fold_model(bg["image"], target_seg_size=seg_size)
                losses = fold_criterion(pred, bg, fold_model.get_router_aux_loss())
                loss = losses["total"] / accum
            
            scaler.scale(loss).backward()
            
            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(fold_model.parameters(), TRAIN_CFG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            ep_loss += losses["total"].item()
            ep_cls += losses["cls"].item()
            ep_seg += losses["seg"].item()
            ep_det += losses["det"].item()
            n_steps += 1
            last_w = {"cls": losses["w_cls"], "seg": losses["w_seg"], "det": losses["w_det"]}
            
            pbar.set_postfix({"loss": f"{ep_loss/n_steps:.3f}", "cls": f"{ep_cls/n_steps:.3f}",
                              "seg": f"{ep_seg/n_steps:.3f}", "det": f"{ep_det/n_steps:.3f}"})
        
        pbar.close()
        scheduler.step()
        avg_tl = ep_loss / max(n_steps, 1)
        cur_lr = optimizer.param_groups[0]["lr"]
        
        # ── Eval ──
        metrics = evaluate_fold(fold_model, val_loader, fold_criterion, seg_size, verbose=True)
        
        hist["train_loss"].append(avg_tl)
        hist["val_loss"].append(metrics["loss"])
        hist["cls_acc"].append(metrics["cls_acc"])
        hist["cls_f1"].append(metrics["cls_f1"])
        hist["seg_dice"].append(metrics["seg_dice"])
        hist["det_ap50"].append(metrics["det_ap50"])
        hist["combined"].append(metrics["combined_weighted"])
        hist["lr"].append(cur_lr)
        hist["cls_loss"].append(ep_cls / max(n_steps, 1))
        hist["seg_loss"].append(ep_seg / max(n_steps, 1))
        hist["det_loss"].append(ep_det / max(n_steps, 1))
        hist["w_cls"].append(last_w["cls"])
        hist["w_seg"].append(last_w["seg"])
        hist["w_det"].append(last_w["det"])
        
        elapsed = time.time() - t0
        star = "🔥" if metrics["combined_weighted"] > best_combined else "  "
        phase = "FROZEN" if epoch < TRAIN_CFG["freeze_epochs"] else "FULL"
        
        print(f"  {star} E{epoch:02d} [{phase}] [{elapsed:.0f}s] lr={cur_lr:.1e} | "
              f"t_loss={avg_tl:.3f} v_loss={metrics['loss']:.3f} | "
              f"cls={metrics['cls_acc']:.1f}% f1={metrics['cls_f1']:.1f}% "
              f"seg={metrics['seg_dice']:.1f}% det_AP50={metrics['det_ap50']:.1f}% | "
              f"comb={metrics['combined_weighted']:.1f}% | "
              f"w[{last_w['cls']:.2f}/{last_w['seg']:.2f}/{last_w['det']:.2f}]")
        
        # Save latest (always)
        save_metrics = {k: v for k, v in metrics.items() if k not in ("cls_preds", "cls_labels")}
        fold_ckpt.save(fold_model, optimizer, scheduler, epoch, fold,
                       {"epoch": epoch, "best_combined": max(best_combined, metrics["combined_weighted"]),
                        "patience_ctr": patience_ctr,
                        "scaler_state": scaler.state_dict(),
                        "history": hist,
                        **save_metrics}, tag="latest")
        
        # Save best
        if metrics["combined_weighted"] > best_combined:
            best_combined = metrics["combined_weighted"]
            patience_ctr = 0
            fold_ckpt.save(fold_model, optimizer, scheduler, epoch, fold,
                           {"epoch": epoch, "best_combined": best_combined, **save_metrics}, tag="best")
            print(f"       💾 New best: {best_combined:.2f}%")
        else:
            patience_ctr += 1
        
        if patience_ctr >= TRAIN_CFG["patience"]:
            print(f"  ⏹️  Early stopping at epoch {epoch}")
            break
    
    # Load best, final eval
    fold_ckpt.load(fold_model, fold=fold, tag="best")
    final = evaluate_fold(fold_model, val_loader, fold_criterion, seg_size)
    
    print(f"\n  ✅ Fold {fold} Best: cls={final['cls_acc']:.1f}% seg={final['seg_dice']:.1f}% "
          f"det_AP50={final['det_ap50']:.1f}% comb={final['combined_weighted']:.1f}%")
    
    del fold_model, optimizer, scheduler, scaler, train_loader, val_loader
    torch.cuda.empty_cache()
    return final, hist


# ──────────────────────────────────────────────────────────────────────────────
# 5. RUN 5-FOLD
# ──────────────────────────────────────────────────────────────────────────────
all_fold_metrics = dict(training_progress.get("completed_folds", {}))
all_fold_histories = dict(training_progress.get("completed_histories", {}))

print(f"\n  🚀 Starting 5-Fold CV for WoundShot-{TRAIN_CFG['variant']}")
if all_fold_metrics:
    print(f"  ↩️  Already completed: folds {sorted(all_fold_metrics.keys())}")
t_start = training_progress.get("start_time", time.time())

for fold in range(TRAIN_CFG["n_folds"]):
    # ── SKIP completed folds ──
    if fold in all_fold_metrics:
        m = all_fold_metrics[fold]
        print(f"\n  ⏩ Fold {fold} SKIPPED (already done: comb={m.get('combined_weighted', 0):.1f}%)")
        continue
    
    fm, fh = train_one_fold(fold, TRAIN_CFG["variant"])
    all_fold_metrics[fold] = fm
    all_fold_histories[fold] = fh
    
    # ── ATOMIC SAVE after each fold ──
    progress_data = {
        "completed_folds": {f: {k: v for k, v in m.items() if k not in ("cls_preds", "cls_labels")}
                           for f, m in all_fold_metrics.items()},
        "completed_histories": all_fold_histories,
        "start_time": t_start,
    }
    _atomic_save(progress_data, PROGRESS_FILE)
    print(f"  💾 Progress saved: {len(all_fold_metrics)}/5 folds done")

total_time = time.time() - t_start

# ──────────────────────────────────────────────────────────────────────────────
# 6. RESULTS TABLE
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  📊 WILLIE-{TRAIN_CFG['variant']} — 5-Fold CV Results")
print(f"{'='*80}")

print(f"\n  {'Fold':<6} {'Cls':>7} {'F1':>7} {'Dice':>7} {'AP@0.5':>7} {'Comb':>7}")
print(f"  {'─'*48}")

accs, f1s, dices, aps, combs = [], [], [], [], []
for fold in range(TRAIN_CFG["n_folds"]):
    m = all_fold_metrics[fold]
    print(f"  {fold:<6} {m['cls_acc']:>6.1f}% {m['cls_f1']:>6.1f}% {m['seg_dice']:>6.1f}% "
          f"{m['det_ap50']:>6.1f}% {m['combined_weighted']:>6.1f}%")
    accs.append(m["cls_acc"]); f1s.append(m["cls_f1"])
    dices.append(m["seg_dice"]); aps.append(m["det_ap50"])
    combs.append(m["combined_weighted"])

print(f"  {'─'*48}")
print(f"  {'Mean':<6} {np.mean(accs):>6.1f}% {np.mean(f1s):>6.1f}% {np.mean(dices):>6.1f}% "
      f"{np.mean(aps):>6.1f}% {np.mean(combs):>6.1f}%")
print(f"  {'Std':<6} {np.std(accs):>6.1f}  {np.std(f1s):>6.1f}  {np.std(dices):>6.1f}  "
      f"{np.std(aps):>6.1f}  {np.std(combs):>6.1f}")
print(f"\n  ⏱️  Total: {total_time/3600:.1f} hours")


# ──────────────────────────────────────────────────────────────────────────────
# 7. VISUALIZATION — 8-Panel Training Dashboard
# ──────────────────────────────────────────────────────────────────────────────
FIG_DIR = CKPT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

colors = plt.cm.Set2(np.linspace(0, 1, 5))

# 7A. Training curves (8 panels)
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle(f"WILLIE-{TRAIN_CFG['variant']} Training Dashboard (5-Fold CV)",
             fontsize=16, fontweight="bold")

panels = [
    ("Train vs Val Loss", "train_loss", "val_loss", None),
    ("Classification Accuracy", "cls_acc", None, 90),
    ("Classification F1 (Macro)", "cls_f1", None, 90),
    ("Segmentation Dice", "seg_dice", None, 90),
    ("Detection AP@0.5", "det_ap50", None, 90),
    ("Combined Weighted (40/40/20)", "combined", None, 95),
    ("Per-Task Loss", None, None, None),  # special
    ("Learned Task Weights", None, None, None),  # special
]

for idx, (title, key1, key2, target) in enumerate(panels):
    ax = axes[idx // 4, idx % 4]
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Epoch")
    
    if idx == 6:  # Per-task loss
        for fold in range(TRAIN_CFG["n_folds"]):
            h = all_fold_histories[fold]
            if fold == 0:
                ax.plot(h["cls_loss"], color="blue", label="cls_loss")
                ax.plot(h["seg_loss"], color="green", label="seg_loss")
                ax.plot(h["det_loss"], color="orange", label="det_loss")
            else:
                ax.plot(h["cls_loss"], color="blue", alpha=0.2)
                ax.plot(h["seg_loss"], color="green", alpha=0.2)
                ax.plot(h["det_loss"], color="orange", alpha=0.2)
        ax.legend(fontsize=8)
    elif idx == 7:  # Task weights
        for fold in range(TRAIN_CFG["n_folds"]):
            h = all_fold_histories[fold]
            if fold == 0:
                ax.plot(h["w_cls"], color="blue", label="w_cls")
                ax.plot(h["w_seg"], color="green", label="w_seg")
                ax.plot(h["w_det"], color="orange", label="w_det")
            else:
                ax.plot(h["w_cls"], color="blue", alpha=0.2)
                ax.plot(h["w_seg"], color="green", alpha=0.2)
                ax.plot(h["w_det"], color="orange", alpha=0.2)
        ax.legend(fontsize=8)
    elif key2:  # Train vs Val
        for fold in range(TRAIN_CFG["n_folds"]):
            h = all_fold_histories[fold]
            ax.plot(h[key1], color=colors[fold], alpha=0.5, linestyle="--")
            ax.plot(h[key2], color=colors[fold], label=f"F{fold}")
        ax.legend(fontsize=7, ncol=2)
    else:
        for fold in range(TRAIN_CFG["n_folds"]):
            h = all_fold_histories[fold]
            ax.plot(h[key1], color=colors[fold], marker=".", ms=3, label=f"F{fold}")
        if target:
            ax.axhline(target, color="red", linestyle=":", alpha=0.7, label=f"Target {target}%")
        ax.set_ylabel("%")
        ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(FIG_DIR / "mini_training_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  📊 Saved: mini_training_dashboard.png")

# 7B. Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(TRAIN_CFG["n_folds"])
w = 0.18
bars = [
    ax.bar(x - 1.5*w, accs, w, label="Cls Acc", color="#2196F3"),
    ax.bar(x - 0.5*w, f1s, w, label="Cls F1", color="#64B5F6"),
    ax.bar(x + 0.5*w, dices, w, label="Seg Dice", color="#4CAF50"),
    ax.bar(x + 1.5*w, aps, w, label="Det AP@0.5", color="#FF9800"),
]
ax.axhline(90, color="red", linestyle=":", lw=2, alpha=0.7, label="Target 90%")
ax.set_xlabel("Fold"); ax.set_ylabel("Score (%)")
ax.set_title(f"WILLIE-{TRAIN_CFG['variant']} — Per-Fold Results", fontsize=14, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([f"Fold {i}" for i in range(TRAIN_CFG["n_folds"])])
ax.legend(fontsize=9); ax.set_ylim(0, 105); ax.grid(True, alpha=0.2, axis="y")
for b_group in bars:
    for bar in b_group:
        ax.annotate(f"{bar.get_height():.1f}", xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=7)
plt.tight_layout()
fig.savefig(FIG_DIR / "mini_fold_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  📊 Saved: mini_fold_bars.png")

# 7C. Confusion matrix
best_fold = int(np.argmax(combs))
bm = all_fold_metrics[best_fold]
if len(bm.get("cls_labels", [])) > 0:
    fig, ax = plt.subplots(figsize=(8, 7))
    cm = confusion_matrix(bm["cls_labels"], bm["cls_preds"])
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"Fold {best_fold} Confusion Matrix — Acc={bm['cls_acc']:.1f}% F1={bm['cls_f1']:.1f}%",
                 fontsize=13, fontweight="bold")
    plt.xticks(rotation=30, ha="right"); plt.tight_layout()
    fig.savefig(FIG_DIR / "mini_confusion.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  📊 Saved: mini_confusion.png")

# 7D. Radar
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
cats = ["Cls Acc", "Cls F1", "Seg Dice", "Det AP@0.5", "Combined"]
vals = [np.mean(accs), np.mean(f1s), np.mean(dices), np.mean(aps), np.mean(combs)]
tgt = [90, 90, 90, 90, 95]
angs = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist()
vals_p, tgt_p = vals + [vals[0]], tgt + [tgt[0]]
angs += angs[:1]
ax.fill(angs, vals_p, alpha=0.25, color="#2196F3")
ax.plot(angs, vals_p, "o-", lw=2, color="#2196F3", label="Achieved")
ax.plot(angs, tgt_p, "s--", lw=2, color="red", alpha=0.7, label="Target")
ax.set_thetagrids(np.degrees(angs[:-1]), cats, fontsize=11)
ax.set_ylim(0, 100)
ax.set_title(f"WILLIE-{TRAIN_CFG['variant']} Mean", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "mini_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  📊 Saved: mini_radar.png")


# ──────────────────────────────────────────────────────────────────────────────
# 8. SAVE RESULTS
# ──────────────────────────────────────────────────────────────────────────────
results = {
    "variant": TRAIN_CFG["variant"], "config": TRAIN_CFG,
    "fold_metrics": {f: {k: v for k, v in m.items() if k not in ("cls_preds", "cls_labels")}
                     for f, m in all_fold_metrics.items()},
    "fold_histories": all_fold_histories,
    "summary": {
        "cls_acc": f"{np.mean(accs):.1f}±{np.std(accs):.1f}",
        "cls_f1": f"{np.mean(f1s):.1f}±{np.std(f1s):.1f}",
        "seg_dice": f"{np.mean(dices):.1f}±{np.std(dices):.1f}",
        "det_ap50": f"{np.mean(aps):.1f}±{np.std(aps):.1f}",
        "combined": f"{np.mean(combs):.1f}±{np.std(combs):.1f}",
    },
    "total_hours": total_time / 3600,
}
torch.save(results, CKPT_DIR / "mini_5fold_results.pt")

met = all(x >= 90 for x in [np.mean(accs), np.mean(dices), np.mean(aps)])
print(f"\n  {'✅ TARGET MET' if met else '⚠️  NOT YET'}: 90%+ per task")
print(f"  Combined: {np.mean(combs):.1f}±{np.std(combs):.1f}% (target 95%)")

print(f"\n{'='*80}")
print(f"  ✅ Cell 3 COMPLETE — WILLIE-MINI")
print(f"  Figures: {FIG_DIR}/")
print(f"  Ready for Cell 4 (Train BASE)")
print(f"{'='*80}")



  Cell 3: Train WILLIE-MINI (5-Fold CV)
  Target: 90%+ per task, 95% combined
    variant: MINI
    n_folds: 5
    epochs: 50
    freeze_epochs: 5
    lr_head: 0.0001
    lr_backbone: 1e-05
    weight_decay: 0.0001
    patience: 12
    grad_clip: 1.0
    batch_size: 4
    accumulation_steps: 2
    seg_size: 512
  ↩️  Resuming: 5 folds already done

  🚀 Starting 5-Fold CV for WILLIE-MINI
  ↩️  Already completed: folds [0, 1, 2, 3, 4]

  ⏩ Fold 0 SKIPPED (already done: comb=81.9%)

  ⏩ Fold 1 SKIPPED (already done: comb=84.7%)

  ⏩ Fold 2 SKIPPED (already done: comb=84.4%)

  ⏩ Fold 3 SKIPPED (already done: comb=87.6%)

  ⏩ Fold 4 SKIPPED (already done: comb=84.2%)

  📊 WILLIE-MINI — 5-Fold CV Results

  Fold       Cls      F1    Dice  AP@0.5    Comb
  ────────────────────────────────────────────────
  0        84.7%   84.8%   79.2%   81.7%   81.9%
  1        90.7%   90.7%   79.7%   82.5%   84.7%
  2        83.3%   82.7%   84.3%   86.9%   84.4%
  3        89.4%   89.2%   85.6%   88.0%  

In [5]:
"""
================================================================================
  09_WILLIE_FUSegNet_CSD_MINI.ipynb — Cell 4
  Test Evaluation (Held-Out Set) + Fold Ensemble
  
  Evaluates best checkpoint from each fold on:
    - Classification: 234 held-out test images (cls_test.csv)
    - Segmentation: FUSeg val set (seg has no held-out test labels)
    - Detection: AP@0.5 from seg→connected components
  
  Ensemble: average logits/masks across 5 fold models
================================================================================
"""

import time
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from tqdm.notebook import tqdm

print(f"\n{'='*80}")
print(f"  Cell 4: Test Evaluation — WILLIE-MINI")
print(f"{'='*80}")

VARIANT = "MINI"
seg_size = 512

# ──────────────────────────────────────────────────────────────────────────────
# 1. LOAD TEST DATA
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  📦 Loading Test Data")
print(f"{'─'*80}")

test_loader = get_test_dataloader(batch_size=4)

# Also build seg-only val loader for seg/det eval
seg_val_only = WoundMultiTaskDataset(
    pd.DataFrame(columns=cls_all.columns),   # no cls
    MANIFESTS["seg_val"],                      # 400 seg val images
    MANIFESTS["det_val"],                      # det val
    get_val_transforms(), IMG_SIZE
)
seg_val_loader = DataLoader(seg_val_only, batch_size=4, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True,
                            collate_fn=collate_multitask)
print(f"  Seg/Det eval: {len(seg_val_only)} samples → {len(seg_val_loader)} batches")


# ──────────────────────────────────────────────────────────────────────────────
# 2. PER-FOLD TEST EVALUATION
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  🧪 Per-Fold Test Evaluation")
print(f"{'─'*80}")

fold_ckpt = CheckpointManager(CKPT_DIR / "mini", "woundshot_mini")
fold_criterion = MultiTaskLossSafe(NUM_CLASSES).to(DEVICE)

# ── ATOMIC RESUME: reload per-fold test progress ──
TEST_PROGRESS = CKPT_DIR / "mini_test_progress.pt"

def _atomic_save_test(data, path):
    tmp = Path(str(path) + ".tmp")
    torch.save(data, tmp)
    tmp.rename(path)

if TEST_PROGRESS.exists():
    _test_prog = torch.load(TEST_PROGRESS, map_location="cpu", weights_only=False)
    fold_test_results = _test_prog.get("fold_results", {})
    all_fold_logits = _test_prog.get("fold_logits", [])
    print(f"  ↩️  Resuming test eval: {len(fold_test_results)} folds already done")
else:
    fold_test_results = {}
    all_fold_logits = []

all_fold_seg_preds = []  # for seg ensemble

for fold in range(5):
    # ── SKIP completed folds ──
    if fold in fold_test_results and fold < len(all_fold_logits):
        m = fold_test_results[fold]
        print(f"\n  ⏩ Fold {fold} SKIPPED (already: cls={m['cls_acc']:.1f}% seg={m['seg_dice']:.1f}%)")
        continue
    
    print(f"\n  Fold {fold}:")
    
    # Load best model
    fold_cfg = get_model_config(VARIANT)
    fold_model = WILLIEModel(fold_cfg).to(DEVICE)
    fold_ckpt.load(fold_model, fold=fold, tag="best")
    fold_model.eval()
    
    # ── Classification test ──
    all_logits, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"    Cls test", leave=False):
            bg = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            pred = fold_model(bg["image"], target_seg_size=seg_size)
            
            labels = bg["cls_label"]
            valid = labels >= 0
            if valid.any():
                all_logits.append(pred["cls_logits"][valid].cpu())
                all_labels.append(labels[valid].cpu())
    
    logits_cat = torch.cat(all_logits, 0)
    labels_cat = torch.cat(all_labels, 0)
    preds_cat = logits_cat.argmax(1).numpy()
    labels_np = labels_cat.numpy()
    
    cls_acc = accuracy_score(labels_np, preds_cat) * 100
    cls_f1 = f1_score(labels_np, preds_cat, average="macro") * 100
    
    all_fold_logits.append(logits_cat)
    
    # ── Segmentation + Detection eval (on seg_val) ──
    all_dices, all_det_ap = [], []
    
    with torch.no_grad():
        for batch in tqdm(seg_val_loader, desc=f"    Seg/Det eval", leave=False):
            bg = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            pred = fold_model(bg["image"], target_seg_size=seg_size)
            
            hm = bg["has_mask"]
            if hm.any():
                sp = pred["seg_mask"][hm]
                st = bg["seg_mask"][hm]
                if st.shape[-2:] != sp.shape[-2:]:
                    st = F.interpolate(st, sp.shape[-2:], mode="bilinear", align_corners=False)
                
                all_dices.extend(compute_seg_dice_batch(sp, st))
                all_det_ap.extend(compute_det_ap50_batch(sp, st))
    
    seg_dice = np.mean(all_dices) * 100 if all_dices else 0.0
    det_ap50 = np.mean(all_det_ap) * 100 if all_det_ap else 0.0
    
    fold_test_results[fold] = {
        "cls_acc": cls_acc, "cls_f1": cls_f1,
        "seg_dice": seg_dice, "det_ap50": det_ap50,
        "cls_preds": preds_cat, "cls_labels": labels_np,
    }
    
    cm = compute_combined_metric(cls_acc, seg_dice, det_ap50)
    
    print(f"    cls={cls_acc:.1f}% f1={cls_f1:.1f}% | seg={seg_dice:.1f}% | "
          f"det_AP50={det_ap50:.1f}% | comb={cm['combined_weighted']:.1f}%")
    
    # ── ATOMIC SAVE after each fold ──
    _atomic_save_test({
        "fold_results": {f: {k: v for k, v in m.items()}
                         for f, m in fold_test_results.items()},
        "fold_logits": all_fold_logits,
    }, TEST_PROGRESS)
    print(f"    💾 Test progress saved: {len(fold_test_results)}/5 folds")
    
    del fold_model
    torch.cuda.empty_cache()


# ──────────────────────────────────────────────────────────────────────────────
# 3. ENSEMBLE (average logits across 5 folds)
# ──────────────────────────────────────────────────────────────────────────────
# ── Reconstruct labels_cat if all folds were skipped ──
if 'labels_cat' not in dir():
    labels_cat = torch.tensor(fold_test_results[0]["cls_labels"])
    print(f"  ↩️  Reconstructed labels_cat from saved results ({len(labels_cat)} samples)")
print(f"\n{'─'*80}")
print(f"  🔗 Ensemble (5-Fold Average)")
print(f"{'─'*80}")

# Classification ensemble
ens_logits = torch.stack(all_fold_logits, 0).mean(0)
ens_preds = ens_logits.argmax(1).numpy()
ens_labels = labels_cat.numpy()

ens_cls_acc = accuracy_score(ens_labels, ens_preds) * 100
ens_cls_f1 = f1_score(ens_labels, ens_preds, average="macro") * 100

# Seg + Det: average across folds
seg_dices_all = [fold_test_results[f]["seg_dice"] for f in range(5)]
det_aps_all = [fold_test_results[f]["det_ap50"] for f in range(5)]
ens_seg = np.mean(seg_dices_all)
ens_det = np.mean(det_aps_all)

ens_cm = compute_combined_metric(ens_cls_acc, ens_seg, ens_det)

print(f"  Ensemble Classification: acc={ens_cls_acc:.1f}% f1={ens_cls_f1:.1f}%")
print(f"  Mean Segmentation Dice:  {ens_seg:.1f}%")
print(f"  Mean Detection AP@0.5:   {ens_det:.1f}%")
print(f"  Combined Weighted:       {ens_cm['combined_weighted']:.1f}%")


# ──────────────────────────────────────────────────────────────────────────────
# 4. RESULTS TABLE
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  📊 WILLIE-MINI — Test Results")
print(f"{'='*80}")

print(f"\n  {'Source':<12} {'Cls Acc':>8} {'Cls F1':>8} {'Seg Dice':>9} {'Det AP50':>9} {'Combined':>9}")
print(f"  {'─'*58}")

for fold in range(5):
    m = fold_test_results[fold]
    c = compute_combined_metric(m["cls_acc"], m["seg_dice"], m["det_ap50"])
    print(f"  Fold {fold:<6} {m['cls_acc']:>7.1f}% {m['cls_f1']:>7.1f}% "
          f"{m['seg_dice']:>8.1f}% {m['det_ap50']:>8.1f}% {c['combined_weighted']:>8.1f}%")

print(f"  {'─'*58}")

accs = [fold_test_results[f]["cls_acc"] for f in range(5)]
f1s = [fold_test_results[f]["cls_f1"] for f in range(5)]
dices = [fold_test_results[f]["seg_dice"] for f in range(5)]
aps = [fold_test_results[f]["det_ap50"] for f in range(5)]
combs_list = [compute_combined_metric(a, d, p)["combined_weighted"] for a, d, p in zip(accs, dices, aps)]

print(f"  {'Mean':<12} {np.mean(accs):>7.1f}% {np.mean(f1s):>7.1f}% "
      f"{np.mean(dices):>8.1f}% {np.mean(aps):>8.1f}% {np.mean(combs_list):>8.1f}%")
print(f"  {'Std':<12} {np.std(accs):>7.1f}  {np.std(f1s):>7.1f}  "
      f"{np.std(dices):>8.1f}  {np.std(aps):>8.1f}  {np.std(combs_list):>8.1f}")
print(f"  {'─'*58}")
print(f"  {'Ensemble':<12} {ens_cls_acc:>7.1f}% {ens_cls_f1:>7.1f}% "
      f"{ens_seg:>8.1f}% {ens_det:>8.1f}% {ens_cm['combined_weighted']:>8.1f}%")


# ──────────────────────────────────────────────────────────────────────────────
# 5. PER-CLASS REPORT
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  📋 Per-Class Report (Ensemble)")
print(f"{'─'*80}")
print(classification_report(ens_labels, ens_preds, target_names=CLASS_NAMES, digits=3))


# ──────────────────────────────────────────────────────────────────────────────
# 6. VISUALIZATION
# ──────────────────────────────────────────────────────────────────────────────
FIG_DIR = CKPT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

# 6A. Confusion matrices: per-fold best + ensemble (2×3 grid)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("WILLIE-MINI — Test Confusion Matrices", fontsize=16, fontweight="bold")

for fold in range(5):
    ax = axes[fold // 3, fold % 3]
    m = fold_test_results[fold]
    cm = confusion_matrix(m["cls_labels"], m["cls_preds"])
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(f"Fold {fold} — Acc={m['cls_acc']:.1f}%", fontsize=11, fontweight="bold")
    ax.tick_params(axis='x', rotation=30)

# Ensemble
ax = axes[1, 2]
cm_ens = confusion_matrix(ens_labels, ens_preds)
disp = ConfusionMatrixDisplay(cm_ens, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap="Greens", values_format="d", colorbar=False)
ax.set_title(f"Ensemble — Acc={ens_cls_acc:.1f}%", fontsize=11, fontweight="bold")
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
fig.savefig(FIG_DIR / "mini_test_confusion_all.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  📊 Saved: mini_test_confusion_all.png")


# 6B. Bar chart: per-fold test results + ensemble
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(6)  # 5 folds + ensemble
labels_x = [f"Fold {i}" for i in range(5)] + ["Ensemble"]
w = 0.2

vals_cls = accs + [ens_cls_acc]
vals_dice = dices + [ens_seg]
vals_ap = aps + [ens_det]

b1 = ax.bar(x - w, vals_cls, w, label="Cls Acc", color="#2196F3")
b2 = ax.bar(x, vals_dice, w, label="Seg Dice", color="#4CAF50")
b3 = ax.bar(x + w, vals_ap, w, label="Det AP@0.5", color="#FF9800")

ax.axhline(90, color="red", ls=":", lw=2, alpha=0.7, label="Target 90%")
ax.set_ylabel("Score (%)"); ax.set_xticks(x); ax.set_xticklabels(labels_x)
ax.set_title("WILLIE-MINI — Test Results (Per-Fold + Ensemble)", fontsize=14, fontweight="bold")
ax.legend(fontsize=10); ax.set_ylim(60, 100); ax.grid(True, alpha=0.2, axis="y")

# Highlight ensemble bars
for bar in [b1[-1], b2[-1], b3[-1]]:
    bar.set_edgecolor("black"); bar.set_linewidth(2)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.annotate(f"{bar.get_height():.1f}", xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=7)

plt.tight_layout()
fig.savefig(FIG_DIR / "mini_test_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  📊 Saved: mini_test_bars.png")


# 6C. Radar: test results
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
cats = ["Cls Acc", "Cls F1", "Seg Dice", "Det AP@0.5", "Combined"]
mean_vals = [np.mean(accs), np.mean(f1s), np.mean(dices), np.mean(aps), np.mean(combs_list)]
ens_vals = [ens_cls_acc, ens_cls_f1, ens_seg, ens_det, ens_cm["combined_weighted"]]
tgt = [90, 90, 90, 90, 95]

angs = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist()
angs += angs[:1]

for label, vals, color in [("Mean", mean_vals, "#2196F3"), ("Ensemble", ens_vals, "#4CAF50")]:
    v = vals + [vals[0]]
    ax.fill(angs, v, alpha=0.15, color=color)
    ax.plot(angs, v, "o-", lw=2, color=color, label=label)
t = tgt + [tgt[0]]
ax.plot(angs, t, "s--", lw=2, color="red", alpha=0.7, label="Target")

ax.set_thetagrids(np.degrees(angs[:-1]), cats, fontsize=11)
ax.set_ylim(0, 100)
ax.set_title("WILLIE-MINI Test — Mean vs Ensemble", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="lower right", fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "mini_test_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  📊 Saved: mini_test_radar.png")


# ──────────────────────────────────────────────────────────────────────────────
# 7. SAVE TEST RESULTS
# ──────────────────────────────────────────────────────────────────────────────
test_results = {
    "variant": VARIANT,
    "fold_results": {f: {k: v for k, v in m.items() if k not in ("cls_preds", "cls_labels")}
                     for f, m in fold_test_results.items()},
    "ensemble": {
        "cls_acc": ens_cls_acc, "cls_f1": ens_cls_f1,
        "seg_dice": ens_seg, "det_ap50": ens_det,
        "combined_weighted": ens_cm["combined_weighted"],
    },
    "mean": {
        "cls_acc": f"{np.mean(accs):.1f}±{np.std(accs):.1f}",
        "seg_dice": f"{np.mean(dices):.1f}±{np.std(dices):.1f}",
        "det_ap50": f"{np.mean(aps):.1f}±{np.std(aps):.1f}",
        "combined": f"{np.mean(combs_list):.1f}±{np.std(combs_list):.1f}",
    },
}
torch.save(test_results, CKPT_DIR / "mini_test_results.pt")

print(f"\n{'='*80}")
print(f"  ✅ Cell 4 COMPLETE — Test Evaluation")
print(f"  Test results: {CKPT_DIR / 'mini_test_results.pt'}")
print(f"  Figures: {FIG_DIR}/")
print(f"  Ready for Cell 5 (TTA)")
print(f"{'='*80}")



  Cell 4: Test Evaluation — WILLIE-MINI

────────────────────────────────────────────────────────────────────────────────
  📦 Loading Test Data
────────────────────────────────────────────────────────────────────────────────

  🧪 Test set:
    Dataset: 234 total (234 cls, 0 seg)
    Test: 234 → 59 batches
    Dataset: 767 total (0 cls, 400 seg)
  Seg/Det eval: 767 samples → 192 batches

────────────────────────────────────────────────────────────────────────────────
  🧪 Per-Fold Test Evaluation
────────────────────────────────────────────────────────────────────────────────
  ↩️  Resuming test eval: 5 folds already done

  ⏩ Fold 0 SKIPPED (already: cls=87.6% seg=78.4%)

  ⏩ Fold 1 SKIPPED (already: cls=88.0% seg=77.0%)

  ⏩ Fold 2 SKIPPED (already: cls=88.5% seg=85.1%)

  ⏩ Fold 3 SKIPPED (already: cls=85.9% seg=84.0%)

  ⏩ Fold 4 SKIPPED (already: cls=86.3% seg=84.2%)
  ↩️  Reconstructed labels_cat from saved results (234 samples)

───────────────────────────────────────────────────

In [7]:
# Quick P/R/F1 for MINI detection — self-contained, runs after Cell 4
import numpy as np, cv2

fold_ckpt_prf = CheckpointManager(CKPT_DIR / "mini", "woundshot_mini")
best_fold = int(np.argmax([fold_test_results[f]["seg_dice"] for f in range(5)]))

fold_cfg = get_model_config("MINI")
fold_model = WILLIEModel(fold_cfg).to(DEVICE)
fold_ckpt_prf.load(fold_model, fold=best_fold, tag="best")
fold_model.eval()

TP, FP, FN = 0, 0, 0
with torch.no_grad():
    for batch in seg_val_loader:
        bg = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        pred = fold_model(bg["image"], target_seg_size=512)
        hm = bg["has_mask"]
        if not hm.any():
            continue
        sp = pred["seg_mask"][hm]
        st = bg["seg_mask"][hm]
        if st.shape[-2:] != sp.shape[-2:]:
            st = F.interpolate(st, sp.shape[-2:], mode="bilinear", align_corners=False)
        
        pred_masks = (torch.sigmoid(sp) > 0.5).float().cpu().numpy()
        gt_masks = st.cpu().numpy()
        
        for i in range(pred_masks.shape[0]):
            pb_list = mask_to_bboxes_eval(pred_masks[i, 0])
            gb_list = mask_to_bboxes_eval(gt_masks[i, 0])
            matched_gt = set()
            for pb in pb_list:
                best_iou, best_j = 0, -1
                for j, gb in enumerate(gb_list):
                    iou = compute_iou(pb, gb)
                    if iou > best_iou:
                        best_iou, best_j = iou, j
                if best_iou >= 0.5 and best_j not in matched_gt:
                    TP += 1; matched_gt.add(best_j)
                else:
                    FP += 1
            FN += len(gb_list) - len(matched_gt)

del fold_model; torch.cuda.empty_cache()

prec = TP / (TP + FP + 1e-6) * 100
rec  = TP / (TP + FN + 1e-6) * 100
f1   = 2 * prec * rec / (prec + rec + 1e-6)
print(f"MINI (Fold {best_fold}) → Prec={prec:.2f}%  Rec={rec:.2f}%  F1={f1:.2f}%")
print(f"TP={TP}  FP={FP}  FN={FN}")

Using cache found in <HOME>/.cache/torch/hub/facebookresearch_dinov2_main


  ✅ Loaded: woundshot_mini_fold2_best.pt (epoch 29)
MINI (Fold 2) → Prec=89.03%  Rec=88.66%  F1=88.84%
TP=430  FP=53  FN=55


In [8]:
"""
================================================================================
  09_WILLIE_FUSegNet_CSD_MINI.ipynb — Cell 5 (v2 FIXED)
  TTA + Final MINI Summary
  
  FIX: Reverse spatial transforms on seg masks before averaging
       hflip→un-hflip, vflip→un-vflip, rot90→un-rot90
================================================================================
"""

import time
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from tqdm.notebook import tqdm

print(f"\n{'='*80}")
print(f"  Cell 5: TTA + Final MINI Summary (FIXED)")
print(f"{'='*80}")

VARIANT = "MINI"
seg_size = 512

# ──────────────────────────────────────────────────────────────────────────────
# 1. TTA DATASET
# ──────────────────────────────────────────────────────────────────────────────
class TTADataset(Dataset):
    def __init__(self, cls_df, seg_df=None, img_size=518):
        self.samples = []
        self.img_size = img_size
        if cls_df is not None and len(cls_df) > 0:
            for _, row in cls_df.iterrows():
                self.samples.append({"path": str(row["image_path"]),
                                     "cls_label": int(row["unified_label"]),
                                     "has_mask": False, "mask_path": None})
        if seg_df is not None and len(seg_df) > 0:
            for _, row in seg_df.iterrows():
                self.samples.append({"path": str(row["img"]), "cls_label": -1,
                                     "has_mask": True, "mask_path": str(row["mask"])})
    
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = cv2.imread(s["path"])
        if img is None: img = np.array(Image.open(s["path"]).convert("RGB"))
        else: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = None
        if s["has_mask"] and s["mask_path"]:
            mask = cv2.imread(s["mask_path"], cv2.IMREAD_GRAYSCALE)
            if mask is None: mask = np.array(Image.open(s["mask_path"]).convert("L"))
            mask = (mask > 127).astype(np.float32)
        return {"image_raw": img, "mask_raw": mask, "cls_label": s["cls_label"],
                "has_mask": s["has_mask"]}


# ──────────────────────────────────────────────────────────────────────────────
# 2. TTA WITH PROPER INVERSE TRANSFORMS
# ──────────────────────────────────────────────────────────────────────────────
tta_transforms = get_tta_transforms(IMG_SIZE)
N_TTA = len(tta_transforms)
print(f"  TTA views: {N_TTA} (original, hflip, vflip, hflip+vflip, rot90)")


def reverse_tta_mask(mask_tensor, view_idx):
    """
    Reverse TTA augmentation on predicted seg mask.
    mask_tensor: (1, 1, H, W)
    view_idx: 0=original, 1=hflip, 2=vflip, 3=hflip+vflip, 4=rot90
    """
    if view_idx == 0:
        return mask_tensor                                  # original — no change
    elif view_idx == 1:
        return torch.flip(mask_tensor, dims=[-1])           # un-hflip
    elif view_idx == 2:
        return torch.flip(mask_tensor, dims=[-2])           # un-vflip
    elif view_idx == 3:
        return torch.flip(mask_tensor, dims=[-2, -1])       # un-hflip-vflip
    elif view_idx == 4:
        return torch.rot90(mask_tensor, k=-1, dims=[-2, -1])  # un-rot90 (rotate back)
    return mask_tensor


@torch.no_grad()
def tta_predict_cls(model, img_raw, transforms):
    """Apply N TTA views, average logits."""
    logits_list = []
    for tfm in transforms:
        aug = tfm(image=img_raw)
        img_t = aug["image"].unsqueeze(0).to(DEVICE)
        pred = model(img_t, target_seg_size=seg_size)
        logits_list.append(pred["cls_logits"].cpu())
    return torch.stack(logits_list, 0).mean(0).squeeze(0)


@torch.no_grad()
def tta_predict_seg(model, img_raw, mask_raw, transforms):
    """
    Apply TTA views, REVERSE transform on predicted mask, THEN average.
    This is the critical fix — masks must be in original orientation before averaging.
    """
    seg_preds = []
    for view_idx, tfm in enumerate(transforms):
        aug = tfm(image=img_raw, mask=mask_raw)
        img_t = aug["image"].unsqueeze(0).to(DEVICE)
        pred = model(img_t, target_seg_size=seg_size)
        seg_logit = pred["seg_mask"].cpu()
        
        # REVERSE the augmentation so all masks are in original orientation
        seg_reversed = reverse_tta_mask(seg_logit, view_idx)
        seg_preds.append(seg_reversed)
    
    # Average in original orientation
    avg_seg = torch.stack(seg_preds, 0).mean(0)
    
    # GT mask (in original orientation)
    gt = torch.from_numpy(mask_raw).float().unsqueeze(0).unsqueeze(0)
    if gt.shape[-2:] != avg_seg.shape[-2:]:
        gt = F.interpolate(gt, avg_seg.shape[-2:], mode="bilinear", align_corners=False)
    
    dice_list = compute_seg_dice_batch(avg_seg, gt)
    ap_list = compute_det_ap50_batch(avg_seg, gt)
    return dice_list[0], ap_list[0]


# ──────────────────────────────────────────────────────────────────────────────
# 3. RUN TTA FOR EACH FOLD
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  🧪 TTA Evaluation (5 folds × {N_TTA} views)")
print(f"{'─'*80}")

fold_ckpt = CheckpointManager(CKPT_DIR / "mini", "woundshot_mini")

cls_tta_ds = TTADataset(cls_test, None, IMG_SIZE)
seg_tta_ds = TTADataset(None, MANIFESTS["seg_val"], IMG_SIZE)
print(f"  Cls test: {len(cls_tta_ds)} images")
print(f"  Seg eval: {len(seg_tta_ds)} images")

# ── ATOMIC RESUME: reload per-fold TTA progress ──
TTA_PROGRESS = CKPT_DIR / "mini_tta_progress.pt"

def _atomic_save_tta(data, path):
    tmp = Path(str(path) + ".tmp")
    torch.save(data, tmp)
    tmp.rename(path)

if TTA_PROGRESS.exists():
    _tta_prog = torch.load(TTA_PROGRESS, map_location="cpu", weights_only=False)
    fold_tta_logits = _tta_prog.get("fold_logits", [])
    fold_tta_cls_results = _tta_prog.get("fold_cls_results", [])
    fold_tta_seg_results = _tta_prog.get("fold_seg_results", [])
    _n_done = len(fold_tta_cls_results)
    print(f"  ↩️  Resuming TTA: {_n_done} folds already done")
else:
    fold_tta_logits = []
    fold_tta_seg_results = []
    fold_tta_cls_results = []
    _n_done = 0

for fold in range(5):
    # ── SKIP completed folds ──
    if fold < _n_done:
        r = fold_tta_cls_results[fold]
        s = fold_tta_seg_results[fold]
        print(f"\n  ⏩ Fold {fold} SKIPPED (already: cls={r['cls_acc']:.1f}% seg={s['seg_dice']:.1f}%)")
        continue
    
    print(f"\n  Fold {fold}:")
    t0 = time.time()
    
    fold_cfg = get_model_config(VARIANT)
    fold_model = WILLIEModel(fold_cfg).to(DEVICE)
    fold_ckpt.load(fold_model, fold=fold, tag="best")
    fold_model.eval()
    
    # ── TTA Classification ──
    logits, labels = [], []
    for i in tqdm(range(len(cls_tta_ds)), desc=f"    TTA Cls", leave=False):
        sample = cls_tta_ds[i]
        l = tta_predict_cls(fold_model, sample["image_raw"], tta_transforms)
        logits.append(l)
        labels.append(sample["cls_label"])
    
    fold_logits = torch.stack(logits, 0)
    fold_labels = torch.tensor(labels)
    fold_preds = fold_logits.argmax(1).numpy()
    fold_labels_np = fold_labels.numpy()
    
    cls_acc = accuracy_score(fold_labels_np, fold_preds) * 100
    cls_f1 = f1_score(fold_labels_np, fold_preds, average="macro") * 100
    fold_tta_logits.append(fold_logits)
    fold_tta_cls_results.append({"cls_acc": cls_acc, "cls_f1": cls_f1})
    
    # ── TTA Segmentation + Detection (with REVERSED masks) ──
    all_dices, all_aps = [], []
    for i in tqdm(range(len(seg_tta_ds)), desc=f"    TTA Seg", leave=False):
        sample = seg_tta_ds[i]
        if sample["has_mask"] and sample["mask_raw"] is not None:
            d, a = tta_predict_seg(fold_model, sample["image_raw"], sample["mask_raw"], tta_transforms)
            all_dices.append(d)
            all_aps.append(a)
    
    seg_dice = np.mean(all_dices) * 100 if all_dices else 0.0
    det_ap50 = np.mean(all_aps) * 100 if all_aps else 0.0
    fold_tta_seg_results.append({"seg_dice": seg_dice, "det_ap50": det_ap50})
    
    elapsed = time.time() - t0
    cm = compute_combined_metric(cls_acc, seg_dice, det_ap50)
    
    print(f"    [{elapsed:.0f}s] cls={cls_acc:.1f}% f1={cls_f1:.1f}% | "
          f"seg={seg_dice:.1f}% | det={det_ap50:.1f}% | comb={cm['combined_weighted']:.1f}%")
    
    # ── ATOMIC SAVE after each fold ──
    _atomic_save_tta({
        "fold_logits": fold_tta_logits,
        "fold_cls_results": fold_tta_cls_results,
        "fold_seg_results": fold_tta_seg_results,
    }, TTA_PROGRESS)
    print(f"    💾 TTA progress saved: {len(fold_tta_cls_results)}/5 folds")
    
    del fold_model
    torch.cuda.empty_cache()


# ──────────────────────────────────────────────────────────────────────────────
# 4. TTA ENSEMBLE
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  🔗 TTA Ensemble (5 folds × {N_TTA} views)")
print(f"{'─'*80}")

tta_ens_logits = torch.stack(fold_tta_logits, 0).mean(0)
tta_ens_preds = tta_ens_logits.argmax(1).numpy()
tta_ens_labels = fold_labels.numpy()

tta_cls_acc = accuracy_score(tta_ens_labels, tta_ens_preds) * 100
tta_cls_f1 = f1_score(tta_ens_labels, tta_ens_preds, average="macro") * 100
tta_seg = np.mean([r["seg_dice"] for r in fold_tta_seg_results])
tta_det = np.mean([r["det_ap50"] for r in fold_tta_seg_results])
tta_cm = compute_combined_metric(tta_cls_acc, tta_seg, tta_det)

print(f"  TTA+Ensemble Classification: acc={tta_cls_acc:.1f}% f1={tta_cls_f1:.1f}%")
print(f"  TTA Mean Seg Dice:           {tta_seg:.1f}%")
print(f"  TTA Mean Det AP@0.5:         {tta_det:.1f}%")
print(f"  TTA Combined Weighted:       {tta_cm['combined_weighted']:.1f}%")


# ──────────────────────────────────────────────────────────────────────────────
# 5. COMPARISON TABLE
# ──────────────────────────────────────────────────────────────────────────────
prev = torch.load(CKPT_DIR / "mini_test_results.pt", weights_only=False)
prev_ens = prev["ensemble"]

print(f"\n{'─'*80}")
print(f"  📊 No-TTA vs TTA Comparison")
print(f"{'─'*80}")
print(f"  {'Metric':<20} {'No-TTA':>10} {'TTA':>10} {'Δ':>8}")
print(f"  {'─'*50}")

comparisons = [
    ("Cls Acc", prev_ens["cls_acc"], tta_cls_acc),
    ("Cls F1", prev_ens["cls_f1"], tta_cls_f1),
    ("Seg Dice", prev_ens["seg_dice"], tta_seg),
    ("Det AP@0.5", prev_ens["det_ap50"], tta_det),
    ("Combined", prev_ens["combined_weighted"], tta_cm["combined_weighted"]),
]
for name, old, new in comparisons:
    d = new - old
    arrow = "↑" if d > 0 else "↓" if d < 0 else "="
    print(f"  {name:<20} {old:>9.1f}% {new:>9.1f}% {arrow}{abs(d):>6.1f}%")


# ──────────────────────────────────────────────────────────────────────────────
# 6. PER-CLASS REPORT
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  📋 Per-Class Report (TTA+Ensemble)")
print(f"{'─'*80}")
print(classification_report(tta_ens_labels, tta_ens_preds, target_names=CLASS_NAMES, digits=3))


# ──────────────────────────────────────────────────────────────────────────────
# 7. VISUALIZATION
# ──────────────────────────────────────────────────────────────────────────────
FIG_DIR = CKPT_DIR / "figures"

# 7A. TTA Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 7))
cm_tta = confusion_matrix(tta_ens_labels, tta_ens_preds)
disp = ConfusionMatrixDisplay(cm_tta, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap="Greens", values_format="d")
ax.set_title(f"WILLIE-MINI TTA+Ensemble — Acc={tta_cls_acc:.1f}% F1={tta_cls_f1:.1f}%",
             fontsize=13, fontweight="bold")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig.savefig(FIG_DIR / "mini_tta_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  📊 Saved: mini_tta_confusion.png")

# 7B. No-TTA vs TTA bar chart
fig, ax = plt.subplots(figsize=(10, 6))
metrics_n = ["Cls Acc", "Cls F1", "Seg Dice", "Det AP@0.5", "Combined"]
no_tta_vals = [prev_ens["cls_acc"], prev_ens["cls_f1"], prev_ens["seg_dice"],
               prev_ens["det_ap50"], prev_ens["combined_weighted"]]
tta_vals = [tta_cls_acc, tta_cls_f1, tta_seg, tta_det, tta_cm["combined_weighted"]]

x = np.arange(len(metrics_n))
w = 0.3
b1 = ax.bar(x - w/2, no_tta_vals, w, label="No-TTA Ensemble", color="#90CAF9")
b2 = ax.bar(x + w/2, tta_vals, w, label="TTA+Ensemble (5 views)", color="#2196F3")
ax.axhline(90, color="red", ls=":", lw=2, alpha=0.7, label="Target 90%")
ax.set_ylabel("Score (%)"); ax.set_xticks(x); ax.set_xticklabels(metrics_n)
ax.set_title("WILLIE-MINI: No-TTA vs TTA", fontsize=14, fontweight="bold")
ax.legend(fontsize=10); ax.set_ylim(60, 100); ax.grid(True, alpha=0.2, axis="y")
for bars in [b1, b2]:
    for bar in bars:
        ax.annotate(f"{bar.get_height():.1f}", xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
plt.tight_layout()
fig.savefig(FIG_DIR / "mini_tta_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  📊 Saved: mini_tta_bars.png")

# 7C. Radar: No-TTA vs TTA vs Target
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
cats = ["Cls Acc", "Cls F1", "Seg Dice", "Det AP@0.5", "Combined"]
tgt = [90, 90, 90, 90, 95]
angs = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist()
angs += angs[:1]

for label, vals, color in [("No-TTA", no_tta_vals, "#90CAF9"), ("TTA", tta_vals, "#2196F3")]:
    v = vals + [vals[0]]
    ax.fill(angs, v, alpha=0.15, color=color)
    ax.plot(angs, v, "o-", lw=2, color=color, label=label)
t = tgt + [tgt[0]]
ax.plot(angs, t, "s--", lw=2, color="red", alpha=0.7, label="Target")

ax.set_thetagrids(np.degrees(angs[:-1]), cats, fontsize=11)
ax.set_ylim(0, 100)
ax.set_title("WILLIE-MINI: No-TTA vs TTA", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="lower right", fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "mini_tta_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  📊 Saved: mini_tta_radar.png")


# ──────────────────────────────────────────────────────────────────────────────
# 8. FINAL MINI SUMMARY
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  📋 FINAL WILLIE-MINI Summary")
print(f"{'='*80}")

print(f"""
  Architecture: DINOv2-ViT-S/14 → FPN(256) → WA-CSA×1 → C+S+D
  Parameters:   34.3M (12.3M trainable, 22.0M frozen backbone)
  P-scSE Seg:   FUSegNet-style parallel channel+spatial SE
  Training:     5-fold CV, 50 epochs, early stopping (patience=12)
  
  ┌─────────────────────────────────────────────────────────────┐
  │ Results         │   No-TTA Ens  │   TTA+Ens     │  Target  │
  ├─────────────────┼───────────────┼───────────────┼──────────┤
  │ Cls Accuracy    │   {prev_ens['cls_acc']:>6.1f}%      │   {tta_cls_acc:>6.1f}%      │   90.0%  │
  │ Cls F1 (macro)  │   {prev_ens['cls_f1']:>6.1f}%      │   {tta_cls_f1:>6.1f}%      │   90.0%  │
  │ Seg Dice        │   {prev_ens['seg_dice']:>6.1f}%      │   {tta_seg:>6.1f}%      │   90.0%  │
  │ Det AP@0.5      │   {prev_ens['det_ap50']:>6.1f}%      │   {tta_det:>6.1f}%      │   90.0%  │
  │ Combined (W)    │   {prev_ens['combined_weighted']:>6.1f}%      │   {tta_cm['combined_weighted']:>6.1f}%      │   95.0%  │
  └─────────────────────────────────────────────────────────────┘
  
  Best per-class: venous=100%, surgical=91.3%
  Weakest:        no_wound=69.3%, diabetic=82.5%
  
  Conclusion: MINI (34M) achieves ~86% combined — strong for edge 
  deployment but below 90% targets. BASE and XL needed for clinical.
""")


# ──────────────────────────────────────────────────────────────────────────────
# 9. SAVE
# ──────────────────────────────────────────────────────────────────────────────
tta_results = {
    "variant": VARIANT,
    "n_tta_views": N_TTA,
    "per_fold_cls": fold_tta_cls_results,
    "per_fold_seg": fold_tta_seg_results,
    "tta_ensemble": {
        "cls_acc": tta_cls_acc, "cls_f1": tta_cls_f1,
        "seg_dice": tta_seg, "det_ap50": tta_det,
        "combined_weighted": tta_cm["combined_weighted"],
    },
    "no_tta_ensemble": prev_ens,
    "improvement": {name: new - old for name, old, new in comparisons},
}
torch.save(tta_results, CKPT_DIR / "mini_tta_results.pt")

print(f"\n{'='*80}")
print(f"  ✅ Cell 5 COMPLETE — TTA + MINI Summary")
print(f"  Notebook 09 DONE. Ready for Notebook 10 (BASE).")
print(f"{'='*80}")



  Cell 5: TTA + Final MINI Summary (FIXED)
  TTA views: 5 (original, hflip, vflip, hflip+vflip, rot90)

────────────────────────────────────────────────────────────────────────────────
  🧪 TTA Evaluation (5 folds × 5 views)
────────────────────────────────────────────────────────────────────────────────
  Cls test: 234 images
  Seg eval: 400 images

  Fold 0:


Using cache found in <HOME>/.cache/torch/hub/facebookresearch_dinov2_main


  ✅ Loaded: woundshot_mini_fold0_best.pt (epoch 9)


    TTA Cls:   0%|          | 0/234 [00:00<?, ?it/s]

    TTA Seg:   0%|          | 0/400 [00:00<?, ?it/s]

...

Using cache found in <HOME>/.cache/torch/hub/facebookresearch_dinov2_main


  ✅ Loaded: woundshot_mini_fold4_best.pt (epoch 27)


    TTA Cls:   0%|          | 0/234 [00:00<?, ?it/s]

    TTA Seg:   0%|          | 0/400 [00:00<?, ?it/s]

    [119s] cls=87.2% f1=86.0% | seg=83.9% | det=86.1% | comb=85.7%
    💾 TTA progress saved: 5/5 folds

────────────────────────────────────────────────────────────────────────────────
  🔗 TTA Ensemble (5 folds × 5 views)
────────────────────────────────────────────────────────────────────────────────
  TTA+Ensemble Classification: acc=88.9% f1=87.4%
  TTA Mean Seg Dice:           81.6%
  TTA Mean Det AP@0.5:         83.5%
  TTA Combined Weighted:       84.9%

────────────────────────────────────────────────────────────────────────────────
  📊 No-TTA vs TTA Comparison
────────────────────────────────────────────────────────────────────────────────
  Metric                   No-TTA        TTA        Δ
  ──────────────────────────────────────────────────
  Cls Acc                   88.9%      88.9% =   0.0%
  Cls F1                    87.6%      87.4% ↓   0.1%
  Seg Dice                  81.8%      81.6% ↓   0.2%
  Det AP@0.5                84.6%      83.5% ↓   1.0%
  Combined           